# PHÂN TÍCH CHI TIẾT: THỜI KHÓA BIỂU

## 1. TỔNG QUAN

### 1.1. Mục đích
Module **Thời Khóa Biểu** quản lý việc xếp lịch học cho từng lớp, giáo viên trong tuần. Module này sử dụng thuật toán **Tabu Search** kết hợp với **Greedy Initialization** để tự động xếp thời khóa biểu tối ưu.

### 1.2. Kiến trúc tổng thể
```
GUI (ThoiKhoaBieu.cs)
    ↓
BUS (ThoiKhoaBieuBUS.cs)
    ↓
SchedulingService (SchedulingService.cs) ← Thuật toán chính
    ↓
DAO (ThoiKhoaBieuDAO.cs)
    ↓
Database (ThoiKhoaBieu, TKB_Temp)
```

### 1.3. Các thành phần chính
- **DTO**: `TimeTableSlotDTO`, `AssignmentSlotDTO`, `ScheduleRequest`, `ScheduleSolution`
- **DAO**: `ThoiKhoaBieuDAO` - Xử lý truy vấn database
- **BUS**: `ThoiKhoaBieuBUS` - Logic nghiệp vụ, validation
- **SchedulingService**: Thuật toán Tabu Search + Greedy
- **TimetableConfigService**: Quản lý cấu hình từ JSON
- **GUI**: `ThoiKhoaBieu` - Giao diện hiển thị và tương tác

---

## 2. KIẾN TRÚC CHI TIẾT

### 2.1. Lớp DTO (Data Transfer Object)

#### 2.1.1. TimeTableSlotDTO
**Vị trí**: `DTO/TimeTableSlotDTO.cs`

**Mục đích**: DTO cho hiển thị từng ô (cell) trên lưới thời khóa biểu.

```csharp
public class TimeTableSlotDTO
{
    public int MaThoiKhoaBieu { get; set; }
    public int MaPhanCong { get; set; }
    public int Thu { get; set; }        // 2-7 (Thứ 2 đến Thứ 7)
    public int Tiet { get; set; }       // 1-10 (Tiết 1 đến Tiết 10)
    public string TenLop { get; set; }
    public string TenMon { get; set; }
    public string TenGiaoVien { get; set; }
    public string MaGiaoVien { get; set; }
    public int MaLop { get; set; }
}
```

**Giải thích**:
- **Thu**: Thứ trong tuần (2 = Thứ 2, 7 = Thứ 7)
- **Tiet**: Tiết học (1-5 = buổi sáng, 6-10 = buổi chiều)
- **TenLop, TenMon, TenGiaoVien**: Đã được join từ các bảng khác để hiển thị

#### 2.1.2. AssignmentSlotDTO
**Vị trí**: `DTO/AssignmentSlotDTO.cs`

**Mục đích**: DTO cho thuật toán scheduling, chỉ chứa thông tin cần thiết.

```csharp
public class AssignmentSlotDTO
{
    public int MaLop { get; set; }
    public int Thu { get; set; }
    public int Tiet { get; set; }
    public int MaMon { get; set; }
    public string MaGV { get; set; }
    public string Phong { get; set; }
}
```

**Giải thích**:
- **Minimal DTO**: Chỉ chứa thông tin cần thiết cho thuật toán (không có tên)
- **Dùng trong memory**: Không cần join với các bảng khác

#### 2.1.3. ScheduleRequest
**Vị trí**: `BUS/Scheduling/Models.cs`

**Mục đích**: Chứa thông tin đầu vào cho thuật toán scheduling.

```csharp
public class ScheduleRequest
{
    public int SemesterId { get; set; }
    public int WeekNo { get; set; }
    public BindingList<int> ClassIds { get; set; }
    public BindingList<string> TeacherIds { get; set; }
    public BindingList<int> SubjectIds { get; set; }
    public BindingList<AssignmentRequirement> Assignments { get; set; }
    public SlotsConfig SlotsConfig { get; set; }
    public WeightConfig WeightConfig { get; set; }
    public int IterMax { get; set; } = 5000;
    public int TabuTenure { get; set; } = 9;
    public int TimeBudgetSec { get; set; } = 90;
    public int NoImproveLimit { get; set; } = 500;
}
```

**Giải thích**:
- **Assignments**: Danh sách yêu cầu (Lớp, Môn, GV, Số tiết/tuần)
- **SlotsConfig**: Cấu hình thời gian (Thứ bắt đầu, thứ kết thúc, số tiết/ngày)
- **WeightConfig**: Trọng số cho các ràng buộc mềm
- **Algorithm parameters**: Số vòng lặp tối đa, độ dài Tabu list, thời gian chạy

#### 2.1.4. ScheduleSolution
**Vị trí**: `BUS/Scheduling/Models.cs`

**Mục đích**: Chứa kết quả của thuật toán scheduling.

```csharp
public class ScheduleSolution
{
    public BindingList<AssignmentSlot> Slots { get; set; }  // Danh sách các tiết đã xếp
    public int Cost { get; set; }                            // Chi phí của giải pháp
    public int HardViolations { get; set; }                  // Số vi phạm ràng buộc cứng
    public SoftCounts SoftCounts { get; set; }              // Số vi phạm ràng buộc mềm
}
```

**Giải thích**:
- **Cost**: Càng thấp càng tốt (0 = hoàn hảo)
- **HardViolations**: Xung đột lớp/giáo viên (phải = 0)
- **SoftCounts**: Vi phạm ràng buộc mềm (có thể chấp nhận)

---

### 2.2. Lớp DAO (Data Access Object)

#### 2.2.1. Cấu trúc cơ bản
**Vị trí**: `DAO/DAOClass/ThoiKhoaBieuDAO.cs`

**Mục đích**: Xử lý tất cả các thao tác với database.

#### 2.2.2. Phương thức đọc dữ liệu

**GetTKBViewByHocKy() - Lấy TKB theo học kỳ với JOIN**

```csharp
public List<TimeTableSlotDTO> GetTKBViewByHocKy(int maHocKy)
{
    const string sql = @"
        SELECT 
            tkb.MaThoiKhoaBieu, 
            CAST(SUBSTRING_INDEX(tkb.ThuTrongTuan, ' ', -1) AS SIGNED) AS Thu, 
            tkb.TietBatDau AS Tiet,
            pc.MaPhanCong, 
            l.TenLop, 
            l.MaLop, 
            mh.TenMonHoc AS TenMon, 
            gv.HoTen AS TenGiaoVien, 
            gv.MaGiaoVien
        FROM ThoiKhoaBieu tkb
        JOIN PhanCongGiangDay pc ON tkb.MaPhanCong = pc.MaPhanCong
        JOIN LopHoc l ON pc.MaLop = l.MaLop
        JOIN MonHoc mh ON pc.MaMonHoc = mh.MaMonHoc
        JOIN GiaoVien gv ON pc.MaGiaoVien = gv.MaGiaoVien
        WHERE pc.MaHocKy = @MaHocKy
        ORDER BY tkb.ThuTrongTuan, tkb.TietBatDau";

    var result = new List<TimeTableSlotDTO>();
    using (var conn = ConnectionDatabase.GetConnection())
    {
        conn.Open();
        using (var cmd = new MySqlCommand(sql, conn))
        {
            cmd.Parameters.AddWithValue("@MaHocKy", maHocKy);
            using (var reader = cmd.ExecuteReader())
            {
                while (reader.Read())
                {
                    var slot = new TimeTableSlotDTO
                    {
                        MaThoiKhoaBieu = reader.GetInt32("MaThoiKhoaBieu"),
                        MaPhanCong = reader.GetInt32("MaPhanCong"),
                        Thu = reader.GetInt32("Thu"),
                        Tiet = reader.GetInt32("Tiet"),
                        TenLop = reader.IsDBNull(reader.GetOrdinal("TenLop")) ? string.Empty : reader.GetString("TenLop"),
                        MaLop = reader.GetInt32("MaLop"),
                        TenMon = reader.IsDBNull(reader.GetOrdinal("TenMon")) ? string.Empty : reader.GetString("TenMon"),
                        TenGiaoVien = reader.IsDBNull(reader.GetOrdinal("TenGiaoVien")) ? string.Empty : reader.GetString("TenGiaoVien"),
                        MaGiaoVien = reader.IsDBNull(reader.GetOrdinal("MaGiaoVien")) ? string.Empty : reader.GetString("MaGiaoVien")
                    };
                    result.Add(slot);
                }
            }
        }
    }
    return result;
}
```

**Giải thích**:
- **Multiple JOINs**: Join 4 bảng để lấy đầy đủ thông tin hiển thị
- **SUBSTRING_INDEX**: Parse "Thu 2" → 2 (integer)
- **IsDBNull check**: Xử lý NULL values an toàn

**GetOfficialSchedule() - Lấy TKB chính thức**

```csharp
public List<AssignmentSlotDTO> GetOfficialSchedule(int semesterId, int? maLop = null)
{
    string sql = @"
        SELECT pc.MaLop, 
               CAST(SUBSTRING_INDEX(tkb.ThuTrongTuan, ' ', -1) AS SIGNED) AS Thu,
               tkb.TietBatDau AS Tiet,
               pc.MaMonHoc AS MaMon,
               pc.MaGiaoVien AS MaGV,
               tkb.PhongHoc AS Phong
        FROM ThoiKhoaBieu tkb
        JOIN PhanCongGiangDay pc ON tkb.MaPhanCong = pc.MaPhanCong
        WHERE pc.MaHocKy = @SemesterId";
    
    if (maLop.HasValue && maLop.Value > 0)
    {
        sql += " AND pc.MaLop = @MaLop";
    }

    // Execute query và map to AssignmentSlotDTO...
}
```

**Giải thích**:
- **Optional filter**: Filter theo lớp nếu có
- **Minimal data**: Chỉ lấy thông tin cần thiết cho thuật toán

#### 2.2.3. Phương thức kiểm tra xung đột

**CheckClassBusy() - Kiểm tra lớp có bận không**

```csharp
public bool CheckClassBusy(int maLop, int thu, int tiet, int excludeId = 0)
{
    const string sql = @"
        SELECT COUNT(*) 
        FROM ThoiKhoaBieu tkb
        JOIN PhanCongGiangDay pc ON tkb.MaPhanCong = pc.MaPhanCong
        WHERE pc.MaLop = @MaLop 
            AND CAST(SUBSTRING_INDEX(tkb.ThuTrongTuan, ' ', -1) AS SIGNED) = @Thu 
            AND tkb.TietBatDau = @Tiet 
            AND (@ExcludeId = 0 OR tkb.MaThoiKhoaBieu != @ExcludeId)";

    using (var conn = ConnectionDatabase.GetConnection())
    {
        conn.Open();
        using (var cmd = new MySqlCommand(sql, conn))
        {
            cmd.Parameters.AddWithValue("@MaLop", maLop);
            cmd.Parameters.AddWithValue("@Thu", thu);
            cmd.Parameters.AddWithValue("@Tiet", tiet);
            cmd.Parameters.AddWithValue("@ExcludeId", excludeId);
            
            long count = (long)cmd.ExecuteScalar();
            return count > 0;
        }
    }
}
```

**Giải thích**:
- **Ràng buộc cứng**: Một lớp không thể có 2 tiết học cùng lúc
- **ExcludeId**: Bỏ qua bản ghi hiện tại khi đang cập nhật

**CheckTeacherBusy() - Kiểm tra giáo viên có bận không**

```csharp
public bool CheckTeacherBusy(string maGV, int thu, int tiet, int excludeId = 0)
{
    const string sql = @"
        SELECT COUNT(*) 
        FROM ThoiKhoaBieu tkb
        JOIN PhanCongGiangDay pc ON tkb.MaPhanCong = pc.MaPhanCong
        WHERE pc.MaGiaoVien = @MaGV 
            AND CAST(SUBSTRING_INDEX(tkb.ThuTrongTuan, ' ', -1) AS SIGNED) = @Thu 
            AND tkb.TietBatDau = @Tiet 
            AND (@ExcludeId = 0 OR tkb.MaThoiKhoaBieu != @ExcludeId)";

    // Execute tương tự...
}
```

**Giải thích**:
- **Ràng buộc cứng**: Một giáo viên không thể dạy 2 lớp cùng lúc

#### 2.2.4. Phương thức lưu dữ liệu

**BulkReplace() - Thay thế toàn bộ TKB của học kỳ**

```csharp
public void BulkReplace(int maHocKy, List<AssignmentSlotDTO> slots)
{
    const string deleteSql = @"
        DELETE tkb FROM ThoiKhoaBieu tkb
        JOIN PhanCongGiangDay pc ON tkb.MaPhanCong = pc.MaPhanCong
        WHERE pc.MaHocKy = @MaHocKy";

    const string insertSql = @"
        INSERT INTO ThoiKhoaBieu(MaPhanCong, ThuTrongTuan, TietBatDau, SoTiet, PhongHoc)
        SELECT pc.MaPhanCong,
            CASE WHEN @Thu IN (2,3,4,5,6,7) THEN CONCAT('Thu ', @Thu) ELSE CAST(@Thu AS CHAR) END,
            @Tiet, 1, @Phong
        FROM PhanCongGiangDay pc
        WHERE pc.MaLop = @MaLop AND pc.MaMonHoc = @MaMon AND pc.MaGiaoVien = @MaGV AND pc.MaHocKy = @MaHocKy
        LIMIT 1";

    using (var conn = ConnectionDatabase.GetConnection())
    {
        conn.Open();
        using (var tx = conn.BeginTransaction())
        {
            try
            {
                // Xóa TKB cũ
                using (var delCmd = new MySqlCommand(deleteSql, conn, tx))
                {
                    delCmd.Parameters.AddWithValue("@MaHocKy", maHocKy);
                    delCmd.ExecuteNonQuery();
                }

                // Ghi TKB mới
                foreach (var slot in slots)
                {
                    using (var insCmd = new MySqlCommand(insertSql, conn, tx))
                    {
                        insCmd.Parameters.AddWithValue("@Thu", slot.Thu);
                        insCmd.Parameters.AddWithValue("@Tiet", slot.Tiet);
                        insCmd.Parameters.AddWithValue("@Phong", string.IsNullOrEmpty(slot.Phong) ? (object)DBNull.Value : slot.Phong);
                        insCmd.Parameters.AddWithValue("@MaLop", slot.MaLop);
                        insCmd.Parameters.AddWithValue("@MaMon", slot.MaMon);
                        insCmd.Parameters.AddWithValue("@MaGV", slot.MaGV);
                        insCmd.Parameters.AddWithValue("@MaHocKy", maHocKy);
                        insCmd.ExecuteNonQuery();
                    }
                }

                tx.Commit();
            }
            catch
            {
                tx.Rollback();
                throw;
            }
        }
    }
}
```

**Giải thích**:
- **Bulk operation**: Xóa tất cả và insert lại (nhanh hơn update từng bản ghi)
- **Transaction**: Đảm bảo tính nhất quán
- **JOIN trong DELETE**: Xóa TKB dựa trên học kỳ thông qua PhanCongGiangDay

---

### 2.3. Lớp BUS (Business Logic)

#### 2.3.1. Cấu trúc cơ bản
**Vị trí**: `BUS/BUSClass/ThoiKhoaBieuBUS.cs`

**Mục đích**: Xử lý logic nghiệp vụ, validation, gọi SchedulingService.

#### 2.3.2. Validation và Business Rules

**ValidateAndMove() - Kiểm tra và di chuyển tiết học**

```csharp
public MoveResult ValidateAndMove(int maPhanCong, int thuMoi, int tietMoi, int currentTkbId = 0)
{
    try
    {
        // a. Lấy thông tin phân công (MaLop, MaGV)
        var phanCong = _phanCongDAO.LayPhanCongTheoMa(maPhanCong);
        if (phanCong == null)
        {
            return MoveResult.Fail("Không tìm thấy phân công giảng dạy!");
        }

        int maLop = phanCong.MaLop;
        string maGV = phanCong.MaGiaoVien;

        // b. Kiểm tra lớp có bận không
        if (_dao.CheckClassBusy(maLop, thuMoi, tietMoi, currentTkbId))
        {
            return MoveResult.Fail($"Lớp {tenLop} đã có tiết học khác vào Thứ {thuMoi}, Tiết {tietMoi}.");
        }

        // c. Kiểm tra giáo viên có bận không
        if (_dao.CheckTeacherBusy(maGV, thuMoi, tietMoi, currentTkbId))
        {
            return MoveResult.Fail($"Giáo viên {tenGV} đang dạy lớp khác vào tiết này.");
        }

        // d. Cả hai kiểm tra đều pass
        return MoveResult.Success("Vị trí này hợp lệ, có thể di chuyển.");
    }
    catch (Exception ex)
    {
        return MoveResult.Fail($"Lỗi khi kiểm tra: {ex.Message}");
    }
}
```

**Giải thích**:
- **Pre-validation**: Kiểm tra trước khi di chuyển (UX tốt hơn)
- **MoveResult pattern**: Trả về kết quả thay vì throw exception
- **Dual check**: Kiểm tra cả lớp và giáo viên

**GetTKBViewByHocKy() - Lấy TKB với Chào Cờ và SHL**

```csharp
public List<TimeTableSlotDTO> GetTKBViewByHocKy(int maHocKy)
{
    var result = _dao.GetTKBViewByHocKy(maHocKy);
    
    // ✅ Tự động thêm Chào Cờ và SHL cho tất cả lớp
    AddChaoCoAndSHL(result, maHocKy);
    
    return result;
}

private void AddChaoCoAndSHL(List<TimeTableSlotDTO> slots, int maHocKy)
{
    // Lấy danh sách tất cả lớp có phân công trong học kỳ này
    var allPhanCong = _phanCongDAO.LayPhanCongTheoHocKy(maHocKy);
    var allLopIds = allPhanCong?.Select(pc => pc.MaLop).Distinct().ToList() ?? new List<int>();
    
    foreach (var maLop in allLopIds)
    {
        // Kiểm tra xem đã có Chào Cờ chưa
        bool hasChaoCo = slots.Any(s => s.MaLop == maLop && s.Thu == 2 && s.Tiet == 1);
        if (!hasChaoCo)
        {
            slots.Add(new TimeTableSlotDTO
            {
                MaThoiKhoaBieu = 0, // Không có trong database
                MaPhanCong = 0,
                Thu = 2, // Thứ 2
                Tiet = 1, // Tiết 1 buổi sáng
                TenLop = tenLop,
                TenMon = "Chào cờ",
                TenGiaoVien = "",
                MaGiaoVien = "",
                MaLop = maLop
            });
        }
        
        // Thêm SHL tương tự...
    }
}
```

**Giải thích**:
- **Virtual slots**: Chào Cờ và SHL không có trong database, được thêm vào runtime
- **LINQ Any()**: Kiểm tra đã có slot chưa
- **Business rule**: Tất cả lớp đều có Chào Cờ (Thứ 2, Tiết 1) và SHL (Thứ 6, Tiết cuối)

---

### 2.4. SchedulingService - Thuật toán chính

#### 2.4.1. Tổng quan thuật toán

**Vị trí**: `BUS/Scheduling/SchedulingService.cs`

**Thuật toán**: **Tabu Search** kết hợp với **Greedy Initialization**

**Quy trình**:
```
1. Greedy Initialization (Khởi tạo ban đầu)
   ↓
2. Tabu Search Optimization (Tối ưu hóa)
   ↓
3. TryAddMissingSlots (Thêm các tiết còn thiếu)
   ↓
4. TryForcePlaceMissingSlots (Ép đặt nếu cần)
   ↓
5. RemoveHardViolations (Loại bỏ xung đột)
```

#### 2.4.2. Greedy Initialization

**InitializeGreedy() - Khởi tạo giải pháp ban đầu**

```csharp
private ScheduleSolution InitializeGreedy(ScheduleRequest request)
{
    var sol = new ScheduleSolution();
    
    // ✅ BƯỚC 1: Thêm Chào cờ (tiết 1 thứ 2) và SHL (tiết cuối thứ 6) cho tất cả lớp
    var allClasses = request.Assignments.Select(a => a.MaLop).Distinct().ToList();
    
    const int MA_MON_CHAO_CO = 0;  // Mã đặc biệt
    const int MA_MON_SHL = -1;
    
    // Thêm Chào cờ vào tiết 1 thứ 2 buổi sáng cho tất cả lớp
    foreach (var maLop in allClasses)
    {
        sol.Slots.Add(new AssignmentSlot
        {
            MaLop = maLop,
            Thu = 2, // Thứ 2
            Tiet = 1, // Tiết 1 buổi sáng
            MaMon = MA_MON_CHAO_CO,
            MaGV = "" // Không có giáo viên
        });
    }
    
    // Thêm SHL vào tiết cuối thứ 6 buổi chính
    foreach (var maLop in allClasses)
    {
        int khoi = GetKhoiForClass(maLop);
        bool isMainSessionMorning = (khoi == 11 || khoi == 12);
        int tietSHL = isMainSessionMorning ? 5 : 10; // Tiết cuối buổi chính
        
        sol.Slots.Add(new AssignmentSlot
        {
            MaLop = maLop,
            Thu = 6,
            Tiet = tietSHL,
            MaMon = MA_MON_SHL,
            MaGV = ""
        });
    }
    
    // ✅ BƯỚC 2: Tạo danh sách các tiết cần xếp
    var periodsToPlace = new BindingList<AssignmentRequirement>();
    foreach (var req in request.Assignments)
    {
        if (IsChaoCo(req.MaMon) || IsSHL(req.MaMon))
            continue;
        
        // Tạo N bản sao cho N tiết/tuần
        for (int i = 0; i < req.SoTietTuan; i++)
        {
            periodsToPlace.Add(req);
        }
    }

    // ✅ BƯỚC 3: Sắp xếp theo độ ưu tiên
    var rand = new Random(42);
    periodsToPlace = new BindingList<AssignmentRequirement>(
        periodsToPlace
            .OrderBy(req => GetSubjectPriority(req.MaMon))  // Môn chính > KHTN/KHXH > Kỹ năng
            .ThenBy(x => rand.Next())  // Shuffle trong cùng độ ưu tiên
            .ToList()
    );
    
    // ✅ BƯỚC 4: Tạo danh sách tất cả slot có sẵn
    var allTimeSlots = new BindingList<(int thu, int tiet)>();
    for (int thu = request.SlotsConfig.ThuBatDau; thu <= request.SlotsConfig.ThuKetThuc; thu++)
    {
        for (int tiet = 1; tiet <= request.SlotsConfig.SoTietMoiNgay; tiet++)
        {
            allTimeSlots.Add((thu, tiet));
        }
    }

    // ✅ BƯỚC 5: Xếp từng tiết theo thứ tự ưu tiên
    foreach (var req in periodsToPlace)
    {
        bool placed = false;
        
        int khoi = GetKhoiForClass(req.MaLop);
        bool isMainSessionMorning = (khoi == 11 || khoi == 12);
        
        // Tạo danh sách candidate slots với điểm ưu tiên
        var candidateSlots = allTimeSlots
            .Where(slot =>
            {
                // ✅ Bỏ qua tiết 1 thứ 2 (đã có Chào cờ)
                if (slot.thu == 2 && slot.tiet == 1) return false;
                
                // ✅ Bỏ qua tiết cuối buổi chính thứ 6 (đã có SHL)
                int tietSHL = isMainSessionMorning ? 5 : 10;
                if (slot.thu == 6 && slot.tiet == tietSHL) return false;
                
                // Check if teacher is busy
                bool teacherBusy = sol.Slots.Any(s => s.MaGV == req.MaGV && s.Thu == slot.thu && s.Tiet == slot.tiet);
                // Check if class is busy
                bool classBusy = sol.Slots.Any(s => s.MaLop == req.MaLop && s.Thu == slot.thu && s.Tiet == slot.tiet);
                return !teacherBusy && !classBusy;
            })
            .OrderBy(slot =>
            {
                // ✅ MỨC ƯU TIÊN: Buổi chính T2-T6 > Buổi chính T7 > Buổi phụ T2-T6 > Buổi phụ T7
                
                bool isMainSession = isMainSessionMorning ? (slot.tiet <= 5) : (slot.tiet >= 6);
                bool isWeekday = (slot.thu >= 2 && slot.thu <= 6);
                bool isSaturday = (slot.thu == 7);
                
                int basePriority = 0;
                if (isMainSession && isWeekday)
                    basePriority = 0; // Mức 1: Buổi chính T2-T6
                else if (isMainSession && isSaturday)
                    basePriority = 1000; // Mức 2: Buổi chính T7
                else if (!isMainSession && isWeekday)
                    basePriority = 2000; // Mức 3: Buổi phụ T2-T6
                else
                    basePriority = 3000; // Mức 4: Buổi phụ T7
                
                // Priority 2: Nếu ở buổi phụ, gom lại thành 1-2 ngày
                int auxiliaryConcentrationBonus = 0;
                if (!isMainSession)
                {
                    // Lấy danh sách các ngày đã có tiết buổi phụ cho môn này
                    var auxiliaryDays = new List<int>();
                    foreach (var existingSlot in sol.Slots.Where(s => s.MaLop == req.MaLop && s.MaMon == req.MaMon))
                    {
                        bool isAuxSlot = isMainSessionMorning ? (existingSlot.Tiet >= 6) : (existingSlot.Tiet <= 5);
                        if (isAuxSlot && !auxiliaryDays.Contains(existingSlot.Thu))
                        {
                            auxiliaryDays.Add(existingSlot.Thu);
                        }
                    }
                    
                    // Nếu đã có tiết buổi phụ ở ngày này, ưu tiên cao (gom vào cùng ngày)
                    if (auxiliaryDays.Contains(slot.thu))
                    {
                        auxiliaryConcentrationBonus = -200;
                    }
                }
                
                // Priority 3: Prefer days with fewer periods of this subject
                string key = $"{req.MaLop}|{req.MaMon}|{slot.thu}";
                int countOnDay = dailyCount.ContainsKey(key) ? dailyCount[key] : 0;
                
                // Priority 4: Ưu tiên đặt consecutive periods trong CÙNG BUỔI
                int consecutiveBonus = 0;
                int periodsOnThisDay = sol.Slots.Count(s => 
                    s.MaLop == req.MaLop && s.MaMon == req.MaMon && s.Thu == slot.thu);
                
                if (periodsOnThisDay > 0 && periodsOnThisDay < 4)
                {
                    var existingPeriods = sol.Slots
                        .Where(s => s.MaLop == req.MaLop && s.MaMon == req.MaMon && s.Thu == slot.thu)
                        .Select(s => s.Tiet)
                        .OrderBy(t => t)
                        .ToList();
                    
                    var testPeriods = existingPeriods.Concat(new[] { slot.tiet }).OrderBy(t => t).ToList();
                    bool wouldBeConsecutive = ArePeriodsConsecutive(testPeriods);
                    
                    string slotSession = GetSessionForPeriod(slot.tiet);
                    bool sameSessionAsExisting = existingPeriods.All(p => GetSessionForPeriod(p) == slotSession);
                    
                    if (wouldBeConsecutive && sameSessionAsExisting)
                    {
                        consecutiveBonus = -50; // Ưu tiên cao cho consecutive
                    }
                }
                
                return basePriority + auxiliaryConcentrationBonus + consecutiveBonus * 10 + countOnDay + periodsOnThisDay;
            })
            .ThenBy(slot => rand.Next())  // Randomize for ties
            .ToList();

        // Try to place in the best candidate slot
        foreach (var slot in candidateSlots)
        {
            sol.Slots.Add(new AssignmentSlot
            {
                MaLop = req.MaLop,
                Thu = slot.thu,
                Tiet = slot.tiet,
                MaMon = req.MaMon,
                MaGV = req.MaGV
            });
            placed = true;
            break;
        }
        
        if (!placed)
        {
            // Log failed...
        }
    }

    return sol;
}
```

**Giải thích chi tiết**:

1. **Thêm Chào Cờ và SHL**:
   - Chào Cờ: Thứ 2, Tiết 1 (tất cả lớp)
   - SHL: Thứ 6, Tiết cuối buổi chính (tùy khối)

2. **Tạo danh sách tiết cần xếp**:
   - Với mỗi phân công có N tiết/tuần → tạo N bản sao
   - Sắp xếp theo độ ưu tiên môn học

3. **Ưu tiên xếp tiết**:
   - **Mức 1**: Buổi chính T2-T6 (ưu tiên cao nhất)
   - **Mức 2**: Buổi chính T7
   - **Mức 3**: Buổi phụ T2-T6
   - **Mức 4**: Buổi phụ T7 (ưu tiên thấp nhất)

4. **Gom tiết buổi phụ**:
   - Ưu tiên gom các tiết buổi phụ vào cùng 1-2 ngày
   - Tránh rải rác nhiều ngày

5. **Consecutive periods**:
   - Ưu tiên đặt các tiết liên tiếp trong cùng buổi
   - Ví dụ: [1, 2, 3] tốt hơn [1, 3, 5]

#### 2.4.3. Tabu Search Optimization

**GenerateSchedule() - Thuật toán Tabu Search**

```csharp
public ScheduleSolution GenerateSchedule(ScheduleRequest request, CancellationToken cancellationToken)
{
    var start = DateTime.UtcNow;
    
    // ✅ BƯỚC 1: Khởi tạo giải pháp ban đầu (Greedy)
    var best = InitializeGreedy(request);
    best.Cost = EvaluateCost(best, request.WeightConfig);
    var bestCost = best.Cost;
    
    // ✅ BƯỚC 2: Khởi tạo Tabu list và các biến
    var tabu = new Dictionary<string, int>();  // Tabu list: move signature → iteration
    var rand = new Random(42);
    var iterSinceImprove = 0;
    
    var stopwatch = Stopwatch.StartNew();
    
    // ✅ BƯỚC 3: Vòng lặp chính (Tabu Search)
    for (int iter = 0; iter < request.IterMax; iter++)
    {
        // Kiểm tra điều kiện dừng
        if (cancellationToken.IsCancellationRequested) break;
        if (stopwatch.Elapsed.TotalSeconds > request.TimeBudgetSec) break;
        
        // ✅ BƯỚC 3.1: Tạo neighborhood (các giải pháp lân cận)
        var neighborhood = GenerateNeighborhood(best, request);
        ScheduleSolution candidate = null;
        int candidateCost = int.MaxValue;
        
        // ✅ BƯỚC 3.2: Tìm candidate tốt nhất trong neighborhood
        foreach (var neighbor in neighborhood)
        {
            // Kiểm tra ràng buộc cứng
            if (!ValidateHardConstraints(neighbor))
                continue;
            
            // Tính move key (signature)
            var moveKey = ComputeMoveKey(neighbor);
            bool isTabu = tabu.ContainsKey(moveKey);
            
            // Tính cost
            var cost = EvaluateCost(neighbor, request.WeightConfig);
            bool aspiration = cost < bestCost;  // Aspiration criterion
            
            // Chấp nhận nếu không tabu hoặc aspiration
            if (!isTabu || aspiration)
            {
                if (cost < candidateCost)
                {
                    candidate = neighbor;
                    candidateCost = cost;
                }
            }
        }
        
        // ✅ BƯỚC 3.3: Nếu không tìm được candidate, tăng counter
        if (candidate == null)
        {
            iterSinceImprove++;
            if (iterSinceImprove > request.NoImproveLimit) break;
            continue;
        }
        
        // ✅ BƯỚC 3.4: Áp dụng candidate
        best = candidate;
        best.Cost = candidateCost;
        
        // ✅ BƯỚC 3.5: Thêm vào Tabu list
        var tabuKey = ComputeMoveKey(best);
        tabu[tabuKey] = iter + request.TabuTenure + rand.Next(0, 3);  // Dynamic tenure
        
        // ✅ BƯỚC 3.6: Xóa các move hết hạn khỏi Tabu list
        var toRemove = new List<string>();
        foreach (var k in tabu.Keys)
        {
            if (tabu[k] <= iter) toRemove.Add(k);
        }
        foreach (var k in toRemove) tabu.Remove(k);
        
        // ✅ BƯỚC 3.7: Cập nhật best cost
        if (best.Cost < bestCost)
        {
            bestCost = best.Cost;
            iterSinceImprove = 0;
        }
        else
        {
            iterSinceImprove++;
            if (iterSinceImprove > request.NoImproveLimit) break;
        }
    }
    
    // ✅ BƯỚC 4: Thêm các tiết còn thiếu
    var beforeAdd = best.Slots.Count;
    best = TryAddMissingSlots(best, request);
    var afterAdd = best.Slots.Count;
    
    // ✅ BƯỚC 5: Ép đặt nếu vẫn còn thiếu
    if (afterAdd < request.Assignments.Sum(a => a.SoTietTuan))
    {
        best = TryForcePlaceMissingSlots(best, request);
    }
    
    // ✅ BƯỚC 6: Loại bỏ xung đột cứng
    best = RemoveHardViolations(best);
    best.Cost = EvaluateCost(best, request.WeightConfig);
    
    stopwatch.Stop();
    return best;
}
```

**Giải thích chi tiết**:

1. **Tabu Search là gì?**
   - **Meta-heuristic**: Thuật toán tìm kiếm cục bộ
   - **Tránh local optima**: Sử dụng Tabu list để tránh quay lại các giải pháp đã thử
   - **Aspiration criterion**: Cho phép vi phạm Tabu nếu tìm được giải pháp tốt hơn best

2. **Các thành phần**:
   - **Neighborhood**: Tập các giải pháp "gần" giải pháp hiện tại
   - **Tabu list**: Danh sách các move bị cấm (trong một số iteration)
   - **Tabu tenure**: Số iteration một move bị cấm
   - **Aspiration**: Cho phép move tabu nếu tốt hơn best

3. **Điều kiện dừng**:
   - Đạt số iteration tối đa (`IterMax`)
   - Hết thời gian (`TimeBudgetSec`)
   - Không cải thiện trong N iteration (`NoImproveLimit`)

**GenerateNeighborhood() - Tạo các giải pháp lân cận**

```csharp
private IEnumerable<ScheduleSolution> GenerateNeighborhood(ScheduleSolution current, ScheduleRequest request)
{
    var list = new List<ScheduleSolution>();
    var slots = current.Slots;
    var rand = new Random();
    int maxNeighbors = Math.Min(100, slots.Count * 2);
    int generated = 0;

    // ✅ Strategy 1: Swap slots within same class
    var byClass = slots.GroupBy(s => s.MaLop).ToList();
    foreach (var classGroup in byClass)
    {
        var classSlots = classGroup.ToList();
        for (int i = 0; i < Math.Min(10, classSlots.Count) && generated < maxNeighbors; i++)
        {
            for (int j = i + 1; j < Math.Min(10, classSlots.Count) && generated < maxNeighbors; j++)
            {
                var a = classSlots[i];
                var b = classSlots[j];
                
                // Only swap if it makes sense
                if (a.Thu == b.Thu && a.MaMon == b.MaMon) continue;

                var clone = Clone(current);
                var slotA = clone.Slots.First(x => x.MaLop == a.MaLop && x.MaMon == a.MaMon && x.MaGV == a.MaGV && x.Thu == a.Thu && x.Tiet == a.Tiet);
                var slotB = clone.Slots.First(x => x.MaLop == b.MaLop && x.MaMon == b.MaMon && x.MaGV == b.MaGV && x.Thu == b.Thu && x.Tiet == b.Tiet);
                
                // Check if swap is valid (no conflicts)
                bool conflictA = clone.Slots.Any(x => x != slotB && (x.MaLop == slotA.MaLop || x.MaGV == slotA.MaGV) && x.Thu == slotB.Thu && x.Tiet == slotB.Tiet);
                bool conflictB = clone.Slots.Any(x => x != slotA && (x.MaLop == slotB.MaLop || x.MaGV == slotB.MaGV) && x.Thu == slotA.Thu && x.Tiet == slotA.Tiet);
                
                if (!conflictA && !conflictB)
                {
                    (slotA.Thu, slotA.Tiet, slotB.Thu, slotB.Tiet) = (slotB.Thu, slotB.Tiet, slotA.Thu, slotA.Tiet);
                    list.Add(clone);
                    generated++;
                }
            }
        }
    }

    // ✅ Strategy 2: Move slot to a different day
    foreach (var s in slots.OrderBy(x => rand.Next()).Take(Math.Min(30, slots.Count)))
    {
        if (generated >= maxNeighbors) break;
        
        // Try moving to a different day
        for (int thu = request.SlotsConfig.ThuBatDau; thu <= request.SlotsConfig.ThuKetThuc; thu++)
        {
            if (thu == s.Thu) continue;
            
            for (int tiet = 1; tiet <= request.SlotsConfig.SoTietMoiNgay; tiet++)
            {
                bool occupied = slots.Any(x => x.Thu == thu && x.Tiet == tiet && (x.MaLop == s.MaLop || x.MaGV == s.MaGV));
                if (occupied) continue;
                
                var clone = Clone(current);
                var target = clone.Slots.First(x => x.MaLop == s.MaLop && x.MaMon == s.MaMon && x.MaGV == s.MaGV && x.Thu == s.Thu && x.Tiet == s.Tiet);
                target.Thu = thu;
                target.Tiet = tiet;
                list.Add(clone);
                generated++;
                break;
            }
        }
    }

    // ✅ Strategy 3: Try to add missing slots
    var coverage = ValidatePeriodCoverage(request, current);
    var missingAssignments = request.Assignments
        .Where(req =>
        {
            string key = $"{req.MaLop}|{req.MaMon}";
            if (coverage.ContainsKey(key))
            {
                var (required, placed) = coverage[key];
                return placed < required;
            }
            return true;
        })
        .Take(5)
        .ToList();

    foreach (var req in missingAssignments)
    {
        if (generated >= maxNeighbors) break;
        
        // Try to add a new slot
        for (int thu = request.SlotsConfig.ThuBatDau; thu <= request.SlotsConfig.ThuKetThuc; thu++)
        {
            if (generated >= maxNeighbors) break;
            
            for (int tiet = 1; tiet <= request.SlotsConfig.SoTietMoiNgay; tiet++)
            {
                bool teacherBusy = slots.Any(s => s.MaGV == req.MaGV && s.Thu == thu && s.Tiet == tiet);
                bool classBusy = slots.Any(s => s.MaLop == req.MaLop && s.Thu == thu && s.Tiet == tiet);
                
                if (!teacherBusy && !classBusy)
                {
                    var clone = Clone(current);
                    clone.Slots.Add(new AssignmentSlot
                    {
                        MaLop = req.MaLop,
                        Thu = thu,
                        Tiet = tiet,
                        MaMon = req.MaMon,
                        MaGV = req.MaGV
                    });
                    list.Add(clone);
                    generated++;
                    break;
                }
            }
        }
    }

    return list;
}
```

**Giải thích**:
- **Neighborhood strategies**: 3 chiến lược tạo giải pháp lân cận
  1. **Swap**: Đổi chỗ 2 tiết cùng lớp
  2. **Move**: Di chuyển 1 tiết sang ngày khác
  3. **Add**: Thêm tiết còn thiếu
- **Conflict checking**: Kiểm tra xung đột trước khi thêm vào neighborhood
- **Limit size**: Giới hạn số lượng neighbor để tránh quá tải

**EvaluateCost() - Tính chi phí giải pháp**

```csharp
public int EvaluateCost(ScheduleSolution sol, WeightConfig w)
{
    var conflicts = AnalyzeConflicts(sol);
    int hard = conflicts.HardViolations * HardPenalty;  // HardPenalty = 1,000,000

    // Calculate soft constraint violations
    int consecutiveHeavy = CalculateConsecutiveHeavy(sol);
    int subjectSpread = CalculateSubjectSpread(sol);
    int dailyBalance = CalculateDailyBalance(sol);
    int stability = 0;

    int soft = w.TrongSoMonNangLienTiep * consecutiveHeavy
        + w.TrongSoTrenMotNgay * subjectSpread
        + w.TrongSoCanBangNgay * dailyBalance
        + w.TrongSoOnDinh * stability;

    sol.SoftCounts = new SoftCounts
    {
        DemMonNangLienTiep = consecutiveHeavy,
        DemPhanBoTrongNgay = subjectSpread,
        DemCanBangNgay = dailyBalance,
        DemOnDinh = stability
    };

    return hard + soft;
}
```

**Giải thích**:
- **Hard constraints**: Xung đột lớp/giáo viên (phải = 0)
- **Soft constraints**: Các ràng buộc mềm (có thể chấp nhận)
  - **Consecutive heavy**: Môn nặng liên tiếp nhiều ngày
  - **Subject spread**: Phân bổ môn trong ngày
  - **Daily balance**: Cân bằng số tiết giữa các ngày
- **Weighted sum**: Tổng có trọng số

**CalculateSubjectSpread() - Tính penalty cho phân bổ môn trong ngày**

```csharp
private int CalculateSubjectSpread(ScheduleSolution sol)
{
    int penalty = 0;
    var byClassSubjectDay = sol.Slots
        .GroupBy(s => new { s.MaLop, s.MaMon, s.Thu })
        .ToList();

    foreach (var group in byClassSubjectDay)
    {
        int periodsOnDay = group.Count();
        var periods = group.Select(s => s.Tiet).OrderBy(t => t).ToList();

        // Xác định khối để biết buổi chính/phụ
        int khoi = GetKhoiForClass(group.Key.MaLop);
        bool isMainSessionMorning = (khoi == 11 || khoi == 12);
        
        // Cho phép đến 4 tiết/ngày
        if (periodsOnDay > 4)
        {
            penalty += (periodsOnDay - 4) * (periodsOnDay - 4);  // Quadratic penalty
        }
        else if (periodsOnDay == 4)
        {
            // Kiểm tra xem 4 tiết có liên tiếp trong CÙNG BUỔI không
            bool isConsecutive = ArePeriodsConsecutive(periods);
            if (!isConsecutive)
            {
                penalty += 10;
            }
        }
        // Tương tự cho 3 tiết, 2 tiết...
        
        // Penalty cho việc rải các tiết TRÁI BUỔI trên nhiều ngày
        var morningPeriods = periods.Where(p => p >= 1 && p <= 5).ToList();
        var afternoonPeriods = periods.Where(p => p >= 6 && p <= 10).ToList();
        
        int auxiliaryCountOnDay = isMainSessionMorning
            ? afternoonPeriods.Count   // 11,12 → chiều là trái buổi
            : morningPeriods.Count;    // 10 → sáng là trái buổi

        if (auxiliaryCountOnDay > 0)
        {
            // Track auxiliary periods per (Lớp, Môn) theo từng ngày
            // Penalty nếu rải trên nhiều ngày
        }
    }

    return penalty;
}
```

**Giải thích**:
- **GroupBy**: Nhóm theo (Lớp, Môn, Ngày)
- **Consecutive check**: Kiểm tra các tiết có liên tiếp trong cùng buổi không
- **Auxiliary penalty**: Phạt nếu rải tiết buổi phụ trên nhiều ngày

**ArePeriodsConsecutive() - Kiểm tra các tiết có liên tiếp không**

```csharp
private bool ArePeriodsConsecutive(IEnumerable<int> periods)
{
    var sorted = periods.OrderBy(p => p).ToList();
    if (sorted.Count <= 1) return true;
    
    // Kiểm tra xem tất cả tiết có trong cùng buổi không
    bool allInMorning = sorted.All(p => p >= 1 && p <= 5);
    bool allInAfternoon = sorted.All(p => p >= 6 && p <= 10);
    
    if (!allInMorning && !allInAfternoon)
    {
        return false;  // Vượt ranh giới buổi
    }
    
    // Kiểm tra liên tiếp trong cùng buổi
    for (int i = 1; i < sorted.Count; i++)
    {
        if (sorted[i] != sorted[i - 1] + 1)
            return false;
    }
    return true;
}
```

**Giải thích**:
- **Session check**: Kiểm tra tất cả tiết có trong cùng buổi không
- **Consecutive check**: Kiểm tra các tiết có liên tiếp không (1, 2, 3)

#### 2.4.4. TryAddMissingSlots - Thêm các tiết còn thiếu

```csharp
private ScheduleSolution TryAddMissingSlots(ScheduleSolution current, ScheduleRequest request)
{
    var coverage = ValidatePeriodCoverage(request, current);
    var allTimeSlots = new List<(int thu, int tiet)>();
    for (int thu = request.SlotsConfig.ThuBatDau; thu <= request.SlotsConfig.ThuKetThuc; thu++)
    {
        for (int tiet = 1; tiet <= request.SlotsConfig.SoTietMoiNgay; tiet++)
        {
            allTimeSlots.Add((thu, tiet));
        }
    }

    var result = Clone(current);
    int added = 0;

    // Group assignments by (Lop, Mon) to avoid duplicates
    var assignmentGroups = request.Assignments
        .GroupBy(a => new { a.MaLop, a.MaMon })
        .Select(g => g.First())
        .ToList();

    // Collect all missing assignments
    var missingAssignments = new List<(AssignmentRequirement req, int missing)>();
    foreach (var req in assignmentGroups)
    {
        string key = $"{req.MaLop}|{req.MaMon}";
        if (coverage.ContainsKey(key))
        {
            var (required, placed) = coverage[key];
            int missing = required - placed;
            if (missing > 0)
            {
                missingAssignments.Add((req, missing));
            }
        }
        else
        {
            missingAssignments.Add((req, req.SoTietTuan));
        }
    }

    // Sort by missing count (try to fill those with fewer missing first)
    missingAssignments = missingAssignments.OrderBy(x => x.missing).ToList();

    // For each incomplete assignment, try to add missing slots
    foreach (var (req, missing) in missingAssignments)
    {
        int remaining = missing;
        int khoi = GetKhoiForClass(req.MaLop);
        bool isMainSessionMorning = (khoi == 11 || khoi == 12);
        
        // Try multiple strategies
        for (int strategy = 0; strategy < 4 && remaining > 0; strategy++)
        {
            IEnumerable<(int thu, int tiet)> candidateSlots;
            
            switch (strategy)
            {
                case 0:
                    // Strategy 1: Prefer main session first
                    candidateSlots = allTimeSlots
                        .Where(slot =>
                        {
                            bool teacherBusy = result.Slots.Any(s => s.MaGV == req.MaGV && s.Thu == slot.thu && s.Tiet == slot.tiet);
                            bool classBusy = result.Slots.Any(s => s.MaLop == req.MaLop && s.Thu == slot.thu && s.Tiet == slot.tiet);
                            return !teacherBusy && !classBusy;
                        })
                        .OrderBy(slot =>
                        {
                            bool isMainSession = isMainSessionMorning ? (slot.tiet <= 5) : (slot.tiet >= 6);
                            return isMainSession ? 0 : 1;
                        })
                        .ThenBy(slot => slot.thu)
                        .ThenBy(slot => slot.tiet);
                    break;
                case 1:
                    // Strategy 2: Prefer auxiliary session, but concentrate on one day
                    candidateSlots = allTimeSlots
                        .Where(slot =>
                        {
                            bool teacherBusy = result.Slots.Any(s => s.MaGV == req.MaGV && s.Thu == slot.thu && s.Tiet == slot.tiet);
                             bool teacherBusy = result.Slots.Any(s => s.MaGV == req.MaGV && s.Thu == slot.thu && s.Tiet == slot.tiet);
                            bool classBusy = result.Slots.Any(s => s.MaLop == req.MaLop && s.Thu == slot.thu && s.Tiet == slot.tiet);
                            if (teacherBusy || classBusy) return false;
                            
                            // Only consider auxiliary session slots
                            bool isMainSession = isMainSessionMorning ? (slot.tiet <= 5) : (slot.tiet >= 6);
                            return !isMainSession;
                        })
                        .OrderBy(slot =>
                        {
                            // Prefer days that already have auxiliary periods (concentrate on one day)
                            int auxiliaryPeriodsOnDay = result.Slots.Count(s => 
                                s.MaLop == req.MaLop && s.Thu == slot.thu && 
                                (isMainSessionMorning ? s.Tiet >= 6 : s.Tiet <= 5));
                            return auxiliaryPeriodsOnDay > 0 ? 0 : 1;
                        })
                        .ThenBy(slot => slot.thu)
                        .ThenBy(slot => slot.tiet);
                    break;
                case 2:
                    // Strategy 3: Random order (fallback)
                    candidateSlots = allTimeSlots
                        .Where(slot =>
                        {
                            bool teacherBusy = result.Slots.Any(s => s.MaGV == req.MaGV && s.Thu == slot.thu && s.Tiet == slot.tiet);
                            bool classBusy = result.Slots.Any(s => s.MaLop == req.MaLop && s.Thu == slot.thu && s.Tiet == slot.tiet);
                            return !teacherBusy && !classBusy;
                        })
                        .OrderBy(slot => new Random().Next());
                    break;
                default:
                    // Strategy 4: Any available slot (last resort)
                    candidateSlots = allTimeSlots
                        .Where(slot =>
                        {
                            bool teacherBusy = result.Slots.Any(s => s.MaGV == req.MaGV && s.Thu == slot.thu && s.Tiet == slot.tiet);
                            bool classBusy = result.Slots.Any(s => s.MaLop == req.MaLop && s.Thu == slot.thu && s.Tiet == slot.tiet);
                            return !teacherBusy && !classBusy;
                        })
                        .OrderBy(slot => slot.thu)
                        .ThenBy(slot => slot.tiet);
                    break;
            }

            foreach (var slot in candidateSlots.Take(remaining))
            {
                bool teacherBusy = result.Slots.Any(s => s.MaGV == req.MaGV && s.Thu == slot.thu && s.Tiet == slot.tiet);
                bool classBusy = result.Slots.Any(s => s.MaLop == req.MaLop && s.Thu == slot.thu && s.Tiet == slot.tiet);
                
                if (!teacherBusy && !classBusy)
                {
                    result.Slots.Add(new AssignmentSlot
                    {
                        MaLop = req.MaLop,
                        Thu = slot.thu,
                        Tiet = slot.tiet,
                        MaMon = req.MaMon,
                        MaGV = req.MaGV
                    });
                    added++;
                    remaining--;
                    if (remaining == 0) break;
                }
            }
            
            if (remaining == 0) break;
        }
    }

    return result;
}
```

**Giải thích**:
- **Multi-strategy approach**: Thử nhiều chiến lược để tìm slot trống
- **Strategy 1**: Ưu tiên buổi chính
- **Strategy 2**: Ưu tiên buổi phụ nhưng gom vào 1 ngày
- **Strategy 3**: Random (fallback)
- **Strategy 4**: Bất kỳ slot nào còn trống

#### 2.4.5. TryForcePlaceMissingSlots - Ép đặt các tiết còn thiếu

```csharp
private ScheduleSolution TryForcePlaceMissingSlots(ScheduleSolution current, ScheduleRequest request)
{
    // Tương tự TryAddMissingSlots nhưng cho phép soft conflicts
    // (chỉ cần lớp HOẶC giáo viên rảnh, không cần cả hai)
    
    var candidateSlots = allTimeSlots
        .Select(slot =>
        {
            bool teacherBusy = result.Slots.Any(s => s.MaGV == req.MaGV && s.Thu == slot.thu && s.Tiet == slot.tiet);
            bool classBusy = result.Slots.Any(s => s.MaLop == req.MaLop && s.Thu == slot.thu && s.Tiet == slot.tiet);
            
            int conflictLevel = 0;
            if (teacherBusy) conflictLevel += 1;
            if (classBusy) conflictLevel += 2;
            
            return new { slot, conflictLevel, teacherBusy, classBusy };
        })
        .Where(x => !x.teacherBusy || !x.classBusy)  // ✅ At least one must be free
        .OrderBy(x => x.conflictLevel)  // Prefer no conflict
        .Take(remaining)
        .ToList();
    
    // Force place với soft conflict nếu cần
}
```

**Giải thích**:
- **Soft conflict**: Cho phép xung đột nhẹ (chỉ 1 trong 2 bận)
- **Last resort**: Chỉ dùng khi không thể xếp đủ tiết bằng cách thông thường

#### 2.4.6. RemoveHardViolations - Loại bỏ xung đột cứng

```csharp
private ScheduleSolution RemoveHardViolations(ScheduleSolution sol)
{
    var result = new ScheduleSolution
    {
        Slots = new BindingList<AssignmentSlot>(),
        Cost = sol.Cost
    };

    // Track slots by class-time and teacher-time
    var classTimeSlots = new Dictionary<string, AssignmentSlot>();  // Key: "{MaLop}-{Thu}-{Tiet}"
    var teacherTimeSlots = new Dictionary<string, AssignmentSlot>();  // Key: "{MaGV}-{Thu}-{Tiet}"

    foreach (var slot in sol.Slots)
    {
        string classKey = $"{slot.MaLop}-{slot.Thu}-{slot.Tiet}";
        string teacherKey = $"{slot.MaGV}-{slot.Thu}-{slot.Tiet}";

        bool classConflict = classTimeSlots.ContainsKey(classKey);
        bool teacherConflict = teacherTimeSlots.ContainsKey(teacherKey);

        if (!classConflict && !teacherConflict)
        {
            // No conflict, add the slot
            result.Slots.Add(slot);
            classTimeSlots[classKey] = slot;
            teacherTimeSlots[teacherKey] = slot;
        }
        // Conflict detected - skip this duplicate slot
    }

    return result;
}
```

**Giải thích**:
- **Dictionary tracking**: Dùng Dictionary để phát hiện duplicate nhanh (O(1))
- **Keep first**: Giữ slot đầu tiên, bỏ các slot trùng lặp

---

### 2.5. TimetableConfigService - Quản lý cấu hình

#### 2.5.1. Load Configuration từ JSON

**Vị trí**: `BUS/Scheduling/TimetableConfigService.cs`

```csharp
public static TimetableConfigRoot Load()
{
    string configPath = GetConfigPath();  // Tìm file timetable_config.json

    // Nếu không có, tạo default
    if (!File.Exists(configPath))
    {
        var defaultConfig = CreateDefaultConfig();
        Save(defaultConfig, configPath);
        return defaultConfig;
    }

    try
    {
        string jsonContent = File.ReadAllText(configPath);
        var config = JsonSerializer.Deserialize<TimetableConfigRoot>(jsonContent);
        return config;
    }
    catch (Exception ex)
    {
        // On error, return default config
        return CreateDefaultConfig();
    }
}
```

**Giải thích**:
- **JSON configuration**: Cấu hình từ file JSON (dễ chỉnh sửa)
- **Default fallback**: Tự động tạo config mặc định nếu không có file

#### 2.5.2. ApplyConfigToRequest - Áp dụng cấu hình

```csharp
public static ScheduleRequest ApplyConfigToRequest(ScheduleRequest request, TimetableConfigRoot config)
{
    // Apply slots config
    if (config.CauHinhTietHoc != null)
    {
        request.SlotsConfig = config.CauHinhTietHoc;  // Thứ bắt đầu, thứ kết thúc, số tiết/ngày
    }

    // Apply weight config
    if (config.CauHinhTrongSo != null)
    {
        request.WeightConfig = config.CauHinhTrongSo;  // Trọng số các ràng buộc mềm
    }

    // Apply algorithm defaults
    if (config.ThamSoThuatToan != null)
    {
        request.IterMax = config.ThamSoThuatToan.SoVongLapToiDa;
        request.TabuTenure = config.ThamSoThuatToan.DoDaiTabu;
        request.TimeBudgetSec = config.ThamSoThuatToan.ThoiGianChayToiDaGiay;
        request.NoImproveLimit = config.ThamSoThuatToan.GioiHanKhongCaiThien;
    }

    return request;
}
```

**Giải thích**:
- **Configuration injection**: Áp dụng cấu hình vào request
- **Flexible**: Có thể override từ UI nếu cần

---

### 2.6. GUI Layer

#### 2.6.1. Cấu trúc cơ bản
**Vị trí**: `GUI/GUIClass/ThoiKhoaBieu/ThoiKhoaBieu.cs`

#### 2.6.2. Hiển thị thời khóa biểu

**LoadTKB() - Tải và hiển thị TKB**

```csharp
private void LoadTKB(int maHocKy, int? maLop = null)
{
    try
    {
        // ✅ Lấy TKB từ BUS
        var slots = tkbBUS.GetTKBViewByHocKy(maHocKy);
        
        // Filter theo lớp nếu có
        if (maLop.HasValue)
        {
            slots = slots.Where(s => s.MaLop == maLop.Value).ToList();
        }
        
        // ✅ Render vào grid
        RenderTKBToGrid(slots);
    }
    catch (Exception ex)
    {
        MessageBox.Show($"Lỗi khi tải TKB: {ex.Message}", "Lỗi",
            MessageBoxButtons.OK, MessageBoxIcon.Error);
    }
}
```

#### 2.6.3. Tự động xếp thời khóa biểu

**BtnSapXepTuDong_Click() - Xếp tự động**

```csharp
private async void BtnSapXepTuDong_Click(object sender, EventArgs e)
{
    try
    {
        int semesterId = GetSelectedSemesterId();
        if (semesterId <= 0)
        {
            MessageBox.Show("Vui lòng chọn học kỳ!", "Thông báo",
                MessageBoxButtons.OK, MessageBoxIcon.Warning);
            return;
        }

        // ✅ Load config từ JSON
        var config = TimetableConfigService.Load();
        
        // ✅ Hiển thị progress dialog
        var progressForm = new ProgressForm();
        progressForm.Show();
        
        var progress = new Progress<string>(msg => progressForm.UpdateProgress(msg));
        var cancellationToken = new CancellationTokenSource();
        
        // ✅ Gọi SchedulingService
        var schedulingService = new SchedulingService();
        var result = await schedulingService.GenerateToTempWithConfigAsync(
            semesterId,
            weekNo: 1,  // Tuần 1
            config,
            cancellationToken.Token,
            progress);
        
        progressForm.Close();
        
        // ✅ Hiển thị kết quả
        if (result.Success)
        {
            MessageBox.Show(result.Message, "Thông báo",
                MessageBoxButtons.OK, MessageBoxIcon.Information);
            
            // Reload TKB
            LoadTKB(semesterId);
        }
        else
        {
            MessageBox.Show(result.Message, "Lỗi",
                MessageBoxButtons.OK, MessageBoxIcon.Error);
        }
    }
    catch (Exception ex)
    {
        MessageBox.Show($"Lỗi: {ex.Message}", "Lỗi",
            MessageBoxButtons.OK, MessageBoxIcon.Error);
    }
}
```

**Giải thích**:
- **Async/await**: Xử lý bất đồng bộ để không block UI
- **Progress reporting**: Hiển thị tiến trình cho người dùng
- **Cancellation token**: Cho phép hủy quá trình

---

## 3. THUẬT TOÁN TABU SEARCH - CHI TIẾT

### 3.1. Tổng quan Tabu Search

**Tabu Search** là một meta-heuristic algorithm được thiết kế để giải quyết các bài toán tối ưu hóa tổ hợp (combinatorial optimization).

#### 3.1.1. Các thành phần chính

1. **Initial Solution (Giải pháp ban đầu)**
   - Sử dụng **Greedy algorithm** để tạo giải pháp ban đầu
   - Giải pháp này có thể không tối ưu nhưng đủ tốt để bắt đầu

2. **Neighborhood (Lân cận)**
   - Tập các giải pháp "gần" giải pháp hiện tại
   - Được tạo bằng các **move operators**: Swap, Move, Add

3. **Tabu List (Danh sách cấm)**
   - Lưu các move đã thử gần đây
   - Ngăn không cho quay lại các giải pháp đã thử (tránh cycle)

4. **Aspiration Criterion (Tiêu chí khát vọng)**
   - Cho phép vi phạm Tabu nếu tìm được giải pháp tốt hơn best
   - Tránh bỏ lỡ cơ hội cải thiện

5. **Stopping Criteria (Điều kiện dừng)**
   - Đạt số iteration tối đa
   - Hết thời gian cho phép
   - Không cải thiện trong N iteration

#### 3.1.2. Pseudocode

```
1. Initialize:
   - best = GreedyInitialization()
   - tabuList = []
   - iterSinceImprove = 0

2. While (not stopping):
   a. Generate neighborhood from current
   b. Find best candidate in neighborhood:
      - Not in tabuList OR aspiration
   c. If candidate found:
      - current = candidate
      - Add move to tabuList
      - If current < best:
          best = current
          iterSinceImprove = 0
      Else:
          iterSinceImprove++
   d. Remove expired moves from tabuList
   e. If iterSinceImprove > limit:
      break

3. Return best
```

### 3.2. Cơ chế hoạt động chi tiết

#### 3.2.1. Greedy Initialization

**Mục đích**: Tạo giải pháp ban đầu nhanh và hợp lý.

**Quy trình**:

1. **Thêm Chào Cờ và SHL**:
   ```
   Với mỗi lớp:
   - Thêm Chào Cờ: Thứ 2, Tiết 1
   - Thêm SHL: Thứ 6, Tiết cuối buổi chính
   ```

2. **Tạo danh sách tiết cần xếp**:
   ```
   Với mỗi phân công (Lớp, Môn, GV, N tiết/tuần):
   - Tạo N bản sao của requirement
   - Sắp xếp theo độ ưu tiên môn học
   ```

3. **Xếp từng tiết**:
   ```
   Với mỗi requirement:
   a. Tạo danh sách candidate slots (tất cả slot có sẵn)
   b. Filter: Bỏ các slot đã bận (lớp hoặc GV)
   c. Sort theo priority:
      - Buổi chính T2-T6 (ưu tiên cao nhất)
      - Buổi chính T7
      - Buổi phụ T2-T6
      - Buổi phụ T7
      - Gom tiết buổi phụ vào cùng ngày
      - Consecutive periods trong cùng buổi
   d. Chọn slot đầu tiên (tốt nhất)
   e. Thêm vào solution
   ```

**Ví dụ cụ thể**:

```
Phân công: Lớp 10A1, Môn Toán, GV001, 3 tiết/tuần

Bước 1: Tạo 3 requirements (Toán-1, Toán-2, Toán-3)

Bước 2: Xếp Toán-1:
- Candidate slots: Tất cả slot T2-T7, Tiết 1-10
- Filter: Bỏ slot đã bận (Chào Cờ T2-T1, các slot GV001 đã dạy, các slot 10A1 đã học)
- Sort: Ưu tiên buổi chính T2-T6 (10A1 là khối 10 → buổi chính = chiều = tiết 6-10)
- Chọn: Thứ 3, Tiết 6 (buổi chính, không conflict)

Bước 3: Xếp Toán-2:
- Candidate slots: Tương tự
- Filter: Bỏ T3-T6 (đã có Toán-1)
- Sort: Ưu tiên consecutive trong cùng buổi
- Chọn: Thứ 3, Tiết 7 (liên tiếp với Toán-1)

Bước 4: Xếp Toán-3:
- Chọn: Thứ 5, Tiết 6 (buổi chính, khác ngày để tránh quá tải)
```

#### 3.2.2. Tabu Search Loop

**Vòng lặp chính**:

```csharp
for (int iter = 0; iter < request.IterMax; iter++)
{
    // 1. Tạo neighborhood (100 neighbors)
    var neighborhood = GenerateNeighborhood(best, request);
    
    // 2. Tìm candidate tốt nhất
    ScheduleSolution candidate = null;
    int candidateCost = int.MaxValue;
    
    foreach (var neighbor in neighborhood)
    {
        // Kiểm tra ràng buộc cứng
        if (!ValidateHardConstraints(neighbor))
            continue;
        
        // Tính move key (signature)
        var moveKey = ComputeMoveKey(neighbor);
        bool isTabu = tabu.ContainsKey(moveKey);
        
        // Tính cost
        var cost = EvaluateCost(neighbor, request.WeightConfig);
        bool aspiration = cost < bestCost;  // Tốt hơn best
        
        // Chấp nhận nếu không tabu hoặc aspiration
        if (!isTabu || aspiration)
        {
            if (cost < candidateCost)
            {
                candidate = neighbor;
                candidateCost = cost;
            }
        }
    }
    
    // 3. Áp dụng candidate
    if (candidate != null)
    {
        best = candidate;
        best.Cost = candidateCost;
        
        // Thêm vào Tabu list
        var tabuKey = ComputeMoveKey(best);
        tabu[tabuKey] = iter + request.TabuTenure + rand.Next(0, 3);
        
        // Cập nhật best cost
        if (best.Cost < bestCost)
        {
            bestCost = best.Cost;
            iterSinceImprove = 0;
        }
        else
        {
            iterSinceImprove++;
        }
    }
    
    // 4. Xóa moves hết hạn
    var toRemove = tabu.Where(kvp => kvp.Value <= iter).Select(kvp => kvp.Key).ToList();
    foreach (var key in toRemove) tabu.Remove(key);
    
    // 5. Kiểm tra điều kiện dừng
    if (iterSinceImprove > request.NoImproveLimit)
        break;
}
```

**Giải thích từng bước**:

1. **GenerateNeighborhood()**:
   - Tạo 100 giải pháp lân cận bằng cách:
     - Swap 2 tiết cùng lớp
     - Move 1 tiết sang ngày khác
     - Add tiết còn thiếu

2. **Tìm candidate tốt nhất**:
   - Chỉ xét các neighbor không vi phạm ràng buộc cứng
   - Bỏ qua các move trong Tabu list (trừ khi aspiration)
   - Chọn neighbor có cost thấp nhất

3. **Aspiration criterion**:
   - Nếu neighbor tốt hơn best → cho phép vi phạm Tabu
   - Tránh bỏ lỡ cơ hội cải thiện

4. **Tabu tenure**:
   - Mỗi move bị cấm trong `TabuTenure + random(0,3)` iterations
   - Dynamic tenure: Thêm random để tránh pattern

5. **Expiration**:
   - Xóa các move hết hạn khỏi Tabu list
   - Giảm kích thước Tabu list

#### 3.2.3. Move Operators

**a) Swap Operator - Đổi chỗ 2 tiết**

```csharp
// Strategy 1: Swap slots within same class
var byClass = slots.GroupBy(s => s.MaLop).ToList();
foreach (var classGroup in byClass)
{
    var classSlots = classGroup.ToList();
    for (int i = 0; i < classSlots.Count; i++)
    {
        for (int j = i + 1; j < classSlots.Count; j++)
        {
            var a = classSlots[i];  // Tiết A: Thứ 3, Tiết 6
            var b = classSlots[j];  // Tiết B: Thứ 5, Tiết 7
            
            // Swap: A → (T5, T7), B → (T3, T6)
            var clone = Clone(current);
            var slotA = clone.Slots.First(x => ...);
            var slotB = clone.Slots.First(x => ...);
            
            // Check conflicts
            bool conflictA = clone.Slots.Any(x => 
                (x.MaLop == slotA.MaLop || x.MaGV == slotA.MaGV) && 
                x.Thu == slotB.Thu && x.Tiet == slotB.Tiet);
            
            if (!conflictA && !conflictB)
            {
                (slotA.Thu, slotA.Tiet, slotB.Thu, slotB.Tiet) = 
                    (slotB.Thu, slotB.Tiet, slotA.Thu, slotA.Tiet);
                list.Add(clone);
            }
        }
    }
}
```

**Ví dụ**:
```
Giải pháp hiện tại:
- Lớp 10A1, Toán, Thứ 3, Tiết 6
- Lớp 10A1, Văn, Thứ 5, Tiết 7

Swap:
- Lớp 10A1, Toán, Thứ 5, Tiết 7  ← Đổi chỗ
- Lớp 10A1, Văn, Thứ 3, Tiết 6  ← Đổi chỗ

Kiểm tra: Không conflict → Chấp nhận
```

**b) Move Operator - Di chuyển 1 tiết**

```csharp
// Strategy 2: Move slot to a different day
foreach (var s in slots)
{
    // Try moving to a different day
    for (int thu = 2; thu <= 7; thu++)
    {
        if (thu == s.Thu) continue;  // Skip same day
        
        for (int tiet = 1; tiet <= 10; tiet++)
        {
            bool occupied = slots.Any(x => 
                x.Thu == thu && x.Tiet == tiet && 
                (x.MaLop == s.MaLop || x.MaGV == s.MaGV));
            
            if (!occupied)
            {
                var clone = Clone(current);
                var target = clone.Slots.First(x => ...);
                target.Thu = thu;  // Di chuyển
                target.Tiet = tiet;
                list.Add(clone);
                break;
            }
        }
    }
}
```

**Ví dụ**:
```
Giải pháp hiện tại:
- Lớp 10A1, Toán, Thứ 3, Tiết 6

Move:
- Lớp 10A1, Toán, Thứ 4, Tiết 6  ← Di chuyển sang Thứ 4

Kiểm tra: Thứ 4, Tiết 6 trống → Chấp nhận
```

**c) Add Operator - Thêm tiết còn thiếu**

```csharp
// Strategy 3: Try to add missing slots
var coverage = ValidatePeriodCoverage(request, current);
var missingAssignments = request.Assignments
    .Where(req => {
        string key = $"{req.MaLop}|{req.MaMon}";
        if (coverage.ContainsKey(key))
        {
            var (required, placed) = coverage[key];
            return placed < required;  // Còn thiếu
        }
        return true;  // Chưa có
    })
    .Take(5)
    .ToList();

foreach (var req in missingAssignments)
{
    for (int thu = 2; thu <= 7; thu++)
    {
        for (int tiet = 1; tiet <= 10; tiet++)
        {
            bool teacherBusy = slots.Any(s => s.MaGV == req.MaGV && s.Thu == thu && s.Tiet == tiet);
            bool classBusy = slots.Any(s => s.MaLop == req.MaLop && s.Thu == thu && s.Tiet == tiet);
            
            if (!teacherBusy && !classBusy)
            {
                var clone = Clone(current);
                clone.Slots.Add(new AssignmentSlot
                {
                    MaLop = req.MaLop,
                    Thu = thu,
                    Tiet = tiet,
                    MaMon = req.MaMon,
                    MaGV = req.MaGV
                });
                list.Add(clone);
                break;
            }
        }
    }
}
```

**Ví dụ**:
```
Yêu cầu: Lớp 10A1, Toán, 3 tiết/tuần
Hiện tại: Chỉ có 2 tiết (Thứ 3-T6, Thứ 5-T7)
Còn thiếu: 1 tiết

Add:
- Lớp 10A1, Toán, Thứ 4, Tiết 6  ← Thêm tiết mới

Kiểm tra: Thứ 4, Tiết 6 trống → Chấp nhận
```

### 3.3. Tính toán Cost (Chi phí)

#### 3.3.1. Hard Constraints (Ràng buộc cứng)

**Vi phạm ràng buộc cứng = Xung đột lớp hoặc giáo viên**

```csharp
public bool ValidateHardConstraints(ScheduleSolution sol)
{
    var teacherAtTime = new HashSet<string>();  // Key: "{MaGV}-{Thu}-{Tiet}"
    var classAtTime = new HashSet<string>();     // Key: "{MaLop}-{Thu}-{Tiet}"

    foreach (var s in sol.Slots)
    {
        string keyTeacher = $"{s.MaGV}|{s.Thu}|{s.Tiet}";
        if (!teacherAtTime.Add(keyTeacher)) 
            return false;  // ❌ Giáo viên bị trùng

        string keyClass = $"{s.MaLop}|{s.Thu}|{s.Tiet}";
        if (!classAtTime.Add(keyClass)) 
            return false;  // ❌ Lớp bị trùng
    }
    return true;  // ✅ Không có xung đột
}
```

**Penalty**:
```csharp
int hard = conflicts.HardViolations * HardPenalty;  // HardPenalty = 1,000,000
```

**Giải thích**:
- **HardPenalty rất lớn**: Đảm bảo giải pháp có xung đột sẽ có cost rất cao
- **Phải = 0**: Giải pháp hợp lệ phải không có xung đột

#### 3.3.2. Soft Constraints (Ràng buộc mềm)

**a) Consecutive Heavy - Môn nặng liên tiếp nhiều ngày**

```csharp
private int CalculateConsecutiveHeavy(ScheduleSolution sol)
{
    int penalty = 0;
    var byClassSubject = sol.Slots
        .GroupBy(s => new { s.MaLop, s.MaMon })
        .ToList();

    foreach (var group in byClassSubject)
    {
        var days = group.Select(s => s.Thu).OrderBy(d => d).ToList();
        for (int i = 1; i < days.Count; i++)
        {
            if (days[i] == days[i - 1] + 1)  // Consecutive days
            {
                int periodsOnDay1 = group.Count(s => s.Thu == days[i - 1]);
                int periodsOnDay2 = group.Count(s => s.Thu == days[i]);
                
                // Penalty nếu quá nhiều tiết trong 2 ngày liên tiếp
                if (periodsOnDay1 >= 3 || periodsOnDay2 >= 3)
                {
                    penalty += (periodsOnDay1 + periodsOnDay2 - 2);
                }
            }
        }
    }
    return penalty;
}
```

**Ví dụ**:
```
Lớp 10A1, Toán:
- Thứ 3: 2 tiết
- Thứ 4: 3 tiết  ← Quá nhiều
- Thứ 5: 2 tiết

Penalty = (2 + 3 - 2) = 3
```

**b) Subject Spread - Phân bổ môn trong ngày**

```csharp
private int CalculateSubjectSpread(ScheduleSolution sol)
{
    int penalty = 0;
    var byClassSubjectDay = sol.Slots
        .GroupBy(s => new { s.MaLop, s.MaMon, s.Thu })
        .ToList();

    foreach (var group in byClassSubjectDay)
    {
        int periodsOnDay = group.Count();
        var periods = group.Select(s => s.Tiet).OrderBy(t => t).ToList();
        
        // Cho phép đến 4 tiết/ngày
        if (periodsOnDay > 4)
        {
            penalty += (periodsOnDay - 4) * (periodsOnDay - 4);  // Quadratic
        }
        else if (periodsOnDay == 4)
        {
            // Kiểm tra consecutive
            bool isConsecutive = ArePeriodsConsecutive(periods);
            if (!isConsecutive)
            {
                penalty += 10;  // Không liên tiếp → penalty
            }
        }
        
        // Penalty cho tiết trái buổi rải rác
        var morningPeriods = periods.Where(p => p >= 1 && p <= 5).ToList();
        var afternoonPeriods = periods.Where(p => p >= 6 && p <= 10).ToList();
        
        // Nếu có tiết ở cả 2 buổi và mỗi buổi < 2 tiết → rời rạc
        if (morningPeriods.Count > 0 && afternoonPeriods.Count > 0)
        {
            if (morningPeriods.Count < 2 && afternoonPeriods.Count < 2)
            {
                penalty += 5;
            }
        }
    }
    
    return penalty;
}
```

**Ví dụ**:
```
Lớp 10A1, Toán, Thứ 3:
- Tiết 1, 3, 5, 7  ← 4 tiết nhưng không liên tiếp
Penalty = 10

Lớp 10A1, Toán, Thứ 3:
- Tiết 1, 2, 3, 4  ← 4 tiết liên tiếp trong cùng buổi
Penalty = 0  ✅
```

**c) Daily Balance - Cân bằng số tiết giữa các ngày**

```csharp
private int CalculateDailyBalance(ScheduleSolution sol)
{
    int penalty = 0;
    var byClassDay = sol.Slots
        .GroupBy(s => new { s.MaLop, s.Thu })
        .ToList();

    var byClass = byClassDay.GroupBy(g => g.Key.MaLop).ToList();
    foreach (var classGroup in byClass)
    {
        var periodsPerDay = classGroup.Select(g => g.Count()).ToList();
        if (periodsPerDay.Count == 0) continue;

        int avg = periodsPerDay.Sum() / periodsPerDay.Count;
        foreach (int count in periodsPerDay)
        {
            int diff = Math.Abs(count - avg);
            if (diff > 2)  // Cho phép chênh lệch 2 tiết
            {
                penalty += diff - 2;
            }
        }
    }
    return penalty;
}
```

**Ví dụ**:
```
Lớp 10A1 trong tuần:
- Thứ 2: 5 tiết
- Thứ 3: 8 tiết  ← Quá nhiều
- Thứ 4: 4 tiết
- Thứ 5: 6 tiết
- Thứ 6: 5 tiết

Trung bình = (5+8+4+6+5)/5 = 5.6
Thứ 3: diff = |8 - 5.6| = 2.4 > 2 → Penalty = 2.4 - 2 = 0.4
```

#### 3.3.3. Tổng hợp Cost

```csharp
public int EvaluateCost(ScheduleSolution sol, WeightConfig w)
{
    var conflicts = AnalyzeConflicts(sol);
    int hard = conflicts.HardViolations * HardPenalty;  // 1,000,000 per violation

    int consecutiveHeavy = CalculateConsecutiveHeavy(sol);
    int subjectSpread = CalculateSubjectSpread(sol);
    int dailyBalance = CalculateDailyBalance(sol);
    int stability = 0;

    int soft = w.TrongSoMonNangLienTiep * consecutiveHeavy
        + w.TrongSoTrenMotNgay * subjectSpread
        + w.TrongSoCanBangNgay * dailyBalance
        + w.TrongSoOnDinh * stability;

    return hard + soft;
}
```

**Ví dụ tính toán**:
```
Giải pháp:
- Hard violations: 0
- Consecutive heavy: 5
- Subject spread: 10
- Daily balance: 3

Weight config:
- TrongSoMonNangLienTiep = 5
- TrongSoTrenMotNgay = 3
- TrongSoCanBangNgay = 2

Cost = (0 * 1,000,000) + (5 * 5) + (3 * 10) + (2 * 3)
     = 0 + 25 + 30 + 6
     = 61
```

---

## 4. MỨC ĐỘ LIÊN QUAN VÀ TƯƠNG TÁC

### 4.1. Liên quan với Phân Công Giảng Dạy

- **Input chính**: Thời khóa biểu lấy dữ liệu từ Phân Công Giảng Dạy
- **GetBySemester()**: `SchedulingService.BuildRequestFromDatabase()` gọi `PhanCongGiangDayBUS.GetBySemester()`
- **Mapping**: Mỗi phân công → N tiết/tuần cần xếp

### 4.2. Liên quan với Lớp học

- **Hiển thị theo lớp**: `GetTKBViewByHocKy()` có thể filter theo lớp
- **Buổi chính/phụ**: Xác định dựa trên khối của lớp (10, 11, 12)

### 4.3. Liên quan với Giáo viên

- **Ràng buộc cứng**: Một giáo viên không thể dạy 2 lớp cùng lúc
- **Hiển thị theo giáo viên**: `GetTKBByTeacher()` - TKB giảng dạy của giáo viên

### 4.4. Liên quan với Môn học

- **Số tiết/tuần**: Lấy từ `MonHoc.SoTiet` và tính toán dựa trên số tuần trong học kỳ
- **Độ ưu tiên**: Môn chính (Toán, Văn, Anh) được ưu tiên xếp trước

---

## 5. LUỒNG XỬ LÝ ĐIỂN HÌNH

### 5.1. Xếp thời khóa biểu tự động

```
1. User click "Sắp xếp tự động" trong GUI
   ↓
2. GUI load config từ JSON (TimetableConfigService.Load())
   ↓
3. GUI gọi SchedulingService.GenerateToTempWithConfigAsync()
   ↓
4. SchedulingService:
   a. BuildRequestFromDatabase():
      - Load phân công từ PhanCongGiangDay
      - Tính số tiết/tuần cho mỗi phân công
      - Tạo ScheduleRequest
   b. ApplyConfigToRequest(): Áp dụng config
   ↓
5. GenerateSchedule():
   a. InitializeGreedy(): Tạo giải pháp ban đầu
   b. Tabu Search loop: Tối ưu hóa
   c. TryAddMissingSlots(): Thêm tiết còn thiếu
   d. TryForcePlaceMissingSlots(): Ép đặt nếu cần
   e. RemoveHardViolations(): Loại bỏ xung đột
   ↓
6. PersistToTemp(): Lưu vào TKB_Temp
   ↓
7. GUI hiển thị kết quả và cho phép user xem preview
   ↓
8. User click "Chấp nhận" → AcceptToOfficial()
   ↓
9. User click "Hủy" → RollbackTemp()
```

### 5.2. Xem thời khóa biểu

```
1. User chọn Học kỳ và Lớp (optional) trong GUI
   ↓
2. GUI gọi ThoiKhoaBieuBUS.GetTKBViewByHocKy(maHocKy)
   ↓
3. BUS:
   a. Gọi DAO.GetTKBViewByHocKy() để lấy TKB từ database
   b. Tự động thêm Chào Cờ và SHL cho tất cả lớp
   ↓
4. Trả về List<TimeTableSlotDTO>
   ↓
5. GUI render vào DataGridView (lưới thời khóa biểu)
```

### 5.3. Di chuyển tiết học thủ công

```
1. User drag & drop tiết học trong GUI
   ↓
2. GUI gọi ValidateAndMove(maPhanCong, thuMoi, tietMoi)
   ↓
3. BUS:
   a. Lấy thông tin phân công (MaLop, MaGV)
   b. Kiểm tra lớp có bận không (CheckClassBusy)
   c. Kiểm tra giáo viên có bận không (CheckTeacherBusy)
   ↓
4. Nếu hợp lệ:
   a. GUI cập nhật UI
   b. User click "Lưu" → SaveTimetableChange()
   c. DAO.UpdateTKB() hoặc InsertTKB()
   ↓
5. Nếu không hợp lệ:
   - Hiển thị thông báo lỗi
   - Không cho phép di chuyển
```

---

## 6. VÍ DỤ THỰC TẾ

### 6.1. Ví dụ: Xếp TKB cho học kỳ 1

**Input**:
- Học kỳ 1 (MaHocKy = 1)
- 20 lớp (10A1, 10A2, ..., 12C5)
- 15 môn học
- 30 giáo viên

**Quy trình**:

1. **Load phân công**:
   ```
   Tổng: 300 phân công
   - Lớp 10A1: Toán (3 tiết), Văn (3 tiết), Anh (3 tiết), ...
   - Lớp 10A2: Toán (3 tiết), Văn (3 tiết), ...
   ```

2. **Greedy Initialization**:
   ```
   Bước 1: Thêm Chào Cờ và SHL
   - 20 lớp × 2 slot = 40 slot
   
   Bước 2: Xếp các môn học
   - Toán (ưu tiên cao): Xếp vào buổi chính T2-T6
   - Văn (ưu tiên cao): Xếp vào buổi chính T2-T6
   - ...
   - Thể dục (ưu tiên thấp): Xếp vào buổi phụ
   
   Kết quả: ~800/900 tiết được xếp (89%)
   ```

3. **Tabu Search**:
   ```
   Iteration 0: Cost = 5000
   Iteration 100: Cost = 3000 (cải thiện)
   Iteration 500: Cost = 1500 (cải thiện)
   Iteration 1000: Cost = 1200 (cải thiện)
   Iteration 2000: Cost = 1200 (không cải thiện)
   ...
   Iteration 3000: Dừng (không cải thiện trong 1000 iter)
   
   Kết quả: ~850/900 tiết được xếp (94%)
   ```

4. **TryAddMissingSlots**:
   ```
   Còn thiếu: 50 tiết
   - Strategy 1: Thêm vào buổi chính → +30 tiết
   - Strategy 2: Thêm vào buổi phụ (gom ngày) → +15 tiết
   - Strategy 3: Random → +5 tiết
   
   Kết quả: ~900/900 tiết được xếp (100%)
   ```

5. **RemoveHardViolations**:
   ```
   Phát hiện: 2 xung đột (duplicate slots)
   - Loại bỏ 2 slot trùng lặp
   
   Kết quả cuối: 898/900 tiết (99.8%)
   ```

### 6.2. Ví dụ: Di chuyển tiết học

**Input**:
- MaPhanCong = 100 (Lớp 10A1, Toán, GV001)
- Vị trí hiện tại: Thứ 3, Tiết 6
- Vị trí mới: Thứ 4, Tiết 6

**Quy trình**:

1. **ValidateAndMove()**:
   ```csharp
   // Lấy thông tin phân công
   var phanCong = phanCongDAO.LayPhanCongTheoMa(100);
   // MaLop = 1, MaGV = "GV001"
   
   // Kiểm tra lớp 10A1 có bận Thứ 4, Tiết 6 không
   if (dao.CheckClassBusy(1, 4, 6))
       return MoveResult.Fail("Lớp 10A1 đã có tiết khác!");
   
   // Kiểm tra GV001 có bận Thứ 4, Tiết 6 không
   if (dao.CheckTeacherBusy("GV001", 4, 6))
       return MoveResult.Fail("GV001 đang dạy lớp khác!");
   
   // ✅ Hợp lệ
   return MoveResult.Success();
   ```

2. **SaveTimetableChange()**:
   ```sql
   UPDATE ThoiKhoaBieu
   SET ThuTrongTuan = 'Thu 4', TietBatDau = 6
   WHERE MaThoiKhoaBieu = 1234
   ```

---

## 7. TỐI ƯU HÓA VÀ BEST PRACTICES

### 7.1. Sử dụng In-Memory Matrices

**❌ Không tốt** (N+1 query problem):
```csharp
foreach (var slot in slots)
{
    bool busy = dao.CheckClassBusy(slot.MaLop, slot.Thu, slot.Tiet);  // Query trong loop!
}
```

**✅ Tốt** (In-memory check):
```csharp
// Build busy matrix một lần
var classBusy = new HashSet<string>();
var teacherBusy = new HashSet<string>();

foreach (var existingSlot in existingSlots)
{
    classBusy.Add($"{existingSlot.MaLop}-{existingSlot.Thu}-{existingSlot.Tiet}");
    teacherBusy.Add($"{existingSlot.MaGV}-{existingSlot.Thu}-{existingSlot.Tiet}");
}

// Check trong memory (O(1))
foreach (var slot in slots)
{
    string key = $"{slot.MaLop}-{slot.Thu}-{slot.Tiet}";
    if (classBusy.Contains(key)) continue;  // O(1) lookup
}
```

### 7.2. Caching Class-to-Grade Mapping

**✅ Tốt**:
```csharp
private Dictionary<int, int> _classToKhoiCache = new Dictionary<int, int>();

private int GetKhoiForClass(int maLop)
{
    if (_classToKhoiCache.ContainsKey(maLop))
        return _classToKhoiCache[maLop];
    
    // Load từ database một lần
    int khoi = lopHocDAO.GetKhoi(maLop);
    _classToKhoiCache[maLop] = khoi;
    return khoi;
}
```

### 7.3. Batch Operations

**✅ Tốt**:
```csharp
// Clear temp và insert trong cùng transaction
using (var tx = conn.BeginTransaction())
{
    ClearTemp(semesterId, weekNo, conn, tx);
    InsertTemp(semesterId, weekNo, slots, conn, tx);
    tx.Commit();
}
```

### 7.4. Progress Reporting

**✅ Tốt**:
```csharp
public async Task<ScheduleGenerationResult> GenerateToTempWithConfigAsync(
    int semesterId,
    int weekNo,
    TimetableConfigRoot config,
    CancellationToken cancellationToken,
    IProgress<string> progress = null)
{
    progress?.Report("Đang tải dữ liệu...");
    // ...
    progress?.Report("Đang khởi tạo...");
    // ...
    progress?.Report("Đang tối ưu hóa...");
    // ...
}
```

---

## 8. KẾT LUẬN

Module **Thời Khóa Biểu** là một module phức tạp sử dụng:

- **Thuật toán Tabu Search**: Meta-heuristic để tối ưu hóa lịch học
- **Greedy Initialization**: Tạo giải pháp ban đầu nhanh và hợp lý
- **Multi-strategy approach**: Nhiều chiến lược để xếp các tiết còn thiếu
- **Hard/Soft constraints**: Ràng buộc cứng (bắt buộc) và mềm (ưu tiên)
- **Configuration-driven**: Cấu hình từ JSON file
- **Progress reporting**: Báo cáo tiến trình cho người dùng

Module này tự động hóa việc xếp thời khóa biểu, giảm thiểu công việc thủ công và đảm bảo tính tối ưu của lịch học.

# PHÂN TÍCH CHI TIẾT: PHÂN CÔNG GIẢNG DẠY

## 1. TỔNG QUAN

### 1.1. Mục đích
Module **Phân Công Giảng Dạy** quản lý việc phân công giáo viên dạy các môn học cho từng lớp trong từng học kỳ. Đây là module trung tâm kết nối giữa Giáo viên, Lớp học, Môn học và Học kỳ.

### 1.2. Kiến trúc tổng thể
```
GUI (PhanCongGiangDay.cs)
    ↓
BUS (PhanCongGiangDayBUS.cs)
    ↓
DAO (PhanCongGiangDayDAO.cs)
    ↓
Database (PhanCongGiangDay, PhanCong_Temp)
```

### 1.3. Các thành phần chính
- **DTO**: `PhanCongGiangDayDTO`, `PhanCongGiangDayViewModel`, `PhanCongCandidateDTO`
- **DAO**: `PhanCongGiangDayDAO` - Xử lý truy vấn database
- **BUS**: `PhanCongGiangDayBUS` - Logic nghiệp vụ, validation
- **Services**: `AssignmentAutoService`, `AssignmentPersistService` - Tự động phân công và lưu trữ
- **GUI**: `PhanCongGiangDay` - Giao diện người dùng

---

## 2. KIẾN TRÚC CHI TIẾT

### 2.1. Lớp DTO (Data Transfer Object)

#### 2.1.1. PhanCongGiangDayDTO
**Vị trí**: `DTO/PhanCongGiangDayDTO.cs`

**Mục đích**: Chuyển dữ liệu giữa các tầng, đảm bảo tính toàn vẹn dữ liệu.

```csharp
public class PhanCongGiangDayDTO
{
    private int maPhanCong;
    private int maLop;
    private string maGiaoVien;
    private int maMonHoc;
    private int maHocKy;
    private DateTime ngayBatDau;
    private DateTime ngayKetThuc;
    
    // Properties với validation
    public int MaPhanCong
    {
        get { return maPhanCong; }
        set
        {
            if (value > 0)
                maPhanCong = value;
            else
                throw new ArgumentException("Mã phân công phải lớn hơn 0");
        }
    }
    
    // Tương tự cho các properties khác...
}
```

**Giải thích**:
- Sử dụng **backing fields** (`private int maPhanCong`) để kiểm soát truy cập
- **Validation trong setter**: Đảm bảo dữ liệu hợp lệ ngay từ khi gán giá trị
- **Ngày kết thúc phải sau ngày bắt đầu**: Logic nghiệp vụ được nhúng trong DTO

#### 2.1.2. PhanCongGiangDayViewModel
**Vị trí**: `DTO/PhanCongGiangDayViewModel.cs`

**Mục đích**: DTO đặc biệt cho hiển thị trên DataGridView, chứa thông tin đã được "làm phẳng" (flattened).

```csharp
public class PhanCongGiangDayViewModel
{
    public int MaPhanCong { get; set; }
    public string GiaoVien { get; set; }      // Tên giáo viên (đã join)
    public string MonHoc { get; set; }         // Tên môn học (đã join)
    public string Lop { get; set; }            // Tên lớp (đã join)
    public string HocKy { get; set; }          // Tên học kỳ (đã join)
    public string ThoiGian { get; set; }       // "dd/MM/yyyy - dd/MM/yyyy"
    
    // Lưu các mã gốc để dùng khi cần
    public string MaGiaoVien { get; set; }
    public int MaMonHoc { get; set; }
    public int MaLop { get; set; }
    public int MaHocKy { get; set; }
}
```

**Giải thích**:
- **ViewModel pattern**: Tách biệt dữ liệu hiển thị và dữ liệu nghiệp vụ
- **Làm phẳng dữ liệu**: Thay vì hiển thị mã (ID), hiển thị tên (đã join từ các bảng khác)
- **Lưu mã gốc**: Để có thể thao tác (sửa, xóa) sau này

---

### 2.2. Lớp DAO (Data Access Object)

#### 2.2.1. Cấu trúc cơ bản
**Vị trí**: `DAO/DAOClass/PhanCongGiangDayDAO.cs`

**Mục đích**: Xử lý tất cả các thao tác với database (CRUD).

#### 2.2.2. Phương thức đọc dữ liệu

**a) DocDSPhanCong() - Đọc tất cả phân công**

```csharp
public List<PhanCongGiangDayDTO> DocDSPhanCong()
{
    List<PhanCongGiangDayDTO> ds = new List<PhanCongGiangDayDTO>();
    string query = @"SELECT MaPhanCong, MaLop, MaGiaoVien, MaMonHoc, MaHocKy, 
                            NgayBatDau, NgayKetThuc 
                     FROM PhanCongGiangDay 
                     ORDER BY MaHocKy DESC, MaLop ASC";

    try
    {
        using (MySqlConnection conn = ConnectionDatabase.GetConnection())
        {
            conn.Open();
            using (MySqlCommand cmd = new MySqlCommand(query, conn))
            {
                using (MySqlDataReader reader = cmd.ExecuteReader())
                {
                    while (reader.Read())
                    {
                        PhanCongGiangDayDTO pc = new PhanCongGiangDayDTO
                        {
                            MaPhanCong = reader.GetInt32("MaPhanCong"),
                            MaLop = reader.GetInt32("MaLop"),
                            MaGiaoVien = reader.GetString("MaGiaoVien"),
                            MaMonHoc = reader.GetInt32("MaMonHoc"),
                            MaHocKy = reader.GetInt32("MaHocKy"),
                            NgayBatDau = reader.GetDateTime("NgayBatDau"),
                            NgayKetThuc = reader.GetDateTime("NgayKetThuc")
                        };
                        ds.Add(pc);
                    }
                }
            }
        }
    }
    catch (Exception ex)
    {
        Console.WriteLine($"Lỗi DocDSPhanCong: {ex.Message}");
        throw;
    }

    return ds;
}
```

**Giải thích**:
- **Using statement**: Đảm bảo connection được đóng tự động (IDisposable pattern)
- **ExecuteReader()**: Dùng cho SELECT trả về nhiều dòng
- **Mapping thủ công**: Chuyển từ DataReader sang DTO
- **ORDER BY**: Sắp xếp theo học kỳ (mới nhất trước) và lớp (tăng dần)

**b) LayPhanCongTheoHocKy() - Lọc theo học kỳ**

```csharp
public List<PhanCongGiangDayDTO> LayPhanCongTheoHocKy(int maHocKy)
{
    List<PhanCongGiangDayDTO> ds = new List<PhanCongGiangDayDTO>();
    string query = @"SELECT MaPhanCong, MaLop, MaGiaoVien, MaMonHoc, MaHocKy, 
                            NgayBatDau, NgayKetThuc 
                     FROM PhanCongGiangDay 
                     WHERE MaHocKy = @MaHocKy";

    try
    {
        using (MySqlConnection conn = ConnectionDatabase.GetConnection())
        {
            conn.Open();
            using (MySqlCommand cmd = new MySqlCommand(query, conn))
            {
                cmd.Parameters.AddWithValue("@MaHocKy", maHocKy);  // ✅ Parameterized query
                using (MySqlDataReader reader = cmd.ExecuteReader())
                {
                    while (reader.Read())
                    {
                        // Mapping tương tự...
                    }
                }
            }
        }
    }
    catch (Exception ex)
    {
        Console.WriteLine($"Lỗi LayPhanCongTheoHocKy: {ex.Message}");
        throw;
    }

    return ds;
}
```

**Giải thích**:
- **Parameterized query**: Tránh SQL Injection, an toàn hơn string concatenation
- **@MaHocKy**: Named parameter trong MySQL
- **AddWithValue()**: Tự động xử lý kiểu dữ liệu

#### 2.2.3. Phương thức thêm dữ liệu

**ThemPhanCong() - Thêm phân công mới**

```csharp
public bool ThemPhanCong(PhanCongGiangDayDTO phanCong)
{
    // ✅ Check for duplicate subject-class-semester before insert
    if (KiemTraTrungLapMonHoc(phanCong.MaLop, phanCong.MaMonHoc, phanCong.MaHocKy))
    {
        throw new InvalidOperationException(
            $"Môn học {phanCong.MaMonHoc} đã được phân công cho lớp {phanCong.MaLop} trong học kỳ {phanCong.MaHocKy}");
    }

    string query = @"INSERT INTO PhanCongGiangDay(MaLop, MaGiaoVien, MaMonHoc, MaHocKy, NgayBatDau, NgayKetThuc) 
                     VALUES(@MaLop, @MaGiaoVien, @MaMonHoc, @MaHocKy, @NgayBatDau, @NgayKetThuc)";
    try
    {
        using (MySqlConnection conn = ConnectionDatabase.GetConnection())
        {
            conn.Open();
            using (MySqlTransaction tx = conn.BeginTransaction())  // ✅ Transaction
            {
                try
                {
                    using (MySqlCommand cmd = new MySqlCommand(query, conn, tx))
                    {
                        cmd.Parameters.AddWithValue("@MaLop", phanCong.MaLop);
                        cmd.Parameters.AddWithValue("@MaGiaoVien", phanCong.MaGiaoVien);
                        cmd.Parameters.AddWithValue("@MaMonHoc", phanCong.MaMonHoc);
                        cmd.Parameters.AddWithValue("@MaHocKy", phanCong.MaHocKy);
                        cmd.Parameters.AddWithValue("@NgayBatDau", phanCong.NgayBatDau);
                        cmd.Parameters.AddWithValue("@NgayKetThuc", phanCong.NgayKetThuc);

                        int result = cmd.ExecuteNonQuery();
                        tx.Commit();  // ✅ Commit inside try block
                        return result > 0;
                    }
                }
                catch
                {
                    tx.Rollback();  // ✅ Rollback in catch block
                    throw;
                }
            }
        }
    }
    catch (Exception ex)
    {
        Console.WriteLine($"Lỗi ThemPhanCong: {ex.Message}");
        throw;
    }
}
```

**Giải thích**:
- **Transaction**: Đảm bảo tính nhất quán dữ liệu (ACID)
- **KiemTraTrungLapMonHoc()**: Kiểm tra ràng buộc nghiệp vụ TRƯỚC khi insert
- **ExecuteNonQuery()**: Dùng cho INSERT/UPDATE/DELETE (không trả về dữ liệu)
- **Commit/Rollback**: Nếu có lỗi, rollback để không lưu dữ liệu không hợp lệ

**InsertBatch() - Thêm nhiều phân công cùng lúc**

```csharp
public void InsertBatch(List<PhanCongGiangDayDTO> list, MySqlTransaction tx)
{
    string query = @"INSERT INTO PhanCongGiangDay(MaLop, MaGiaoVien, MaMonHoc, MaHocKy, NgayBatDau, NgayKetThuc)
                     VALUES(@MaLop, @MaGiaoVien, @MaMonHoc, @MaHocKy, @NgayBatDau, @NgayKetThuc)";
    string checkQuery = @"SELECT COUNT(*) FROM PhanCongGiangDay 
                         WHERE MaLop = @MaLop 
                         AND MaMonHoc = @MaMonHoc 
                         AND MaHocKy = @MaHocKy";
    var conn = tx.Connection;
    
    foreach (var pc in list)
    {
        // ✅ Check for duplicate subject-class-semester before insert
        using (var checkCmd = new MySqlCommand(checkQuery, conn, tx))
        {
            checkCmd.Parameters.AddWithValue("@MaLop", pc.MaLop);
            checkCmd.Parameters.AddWithValue("@MaMonHoc", pc.MaMonHoc);
            checkCmd.Parameters.AddWithValue("@MaHocKy", pc.MaHocKy);
            int count = Convert.ToInt32(checkCmd.ExecuteScalar());
            if (count > 0)
            {
                throw new InvalidOperationException(
                    $"Môn học {pc.MaMonHoc} đã được phân công cho lớp {pc.MaLop} trong học kỳ {pc.MaHocKy}");
            }
        }

        using (var cmd = new MySqlCommand(query, conn, tx))
        {
            // Set parameters...
            cmd.ExecuteNonQuery();
        }
    }
}
```

**Giải thích**:
- **Batch insert**: Xử lý nhiều bản ghi trong cùng một transaction
- **ExecuteScalar()**: Dùng cho SELECT trả về 1 giá trị (COUNT)
- **Shared transaction**: Tất cả các insert trong cùng 1 transaction, nếu 1 lỗi thì tất cả rollback

#### 2.2.4. Phương thức kiểm tra ràng buộc

**KiemTraTrungLapMonHoc() - Kiểm tra trùng lặp môn học**

```csharp
public bool KiemTraTrungLapMonHoc(int maLop, int maMonHoc, int maHocKy, int? maPhanCongExclude = null)
{
    string query = @"SELECT COUNT(*) FROM PhanCongGiangDay 
                    WHERE MaLop = @MaLop 
                    AND MaMonHoc = @MaMonHoc 
                    AND MaHocKy = @MaHocKy";

    // Nếu đang cập nhật, bỏ qua bản ghi hiện tại
    if (maPhanCongExclude.HasValue)
    {
        query += " AND MaPhanCong != @MaPhanCongExclude";
    }

    try
    {
        using (MySqlConnection conn = ConnectionDatabase.GetConnection())
        {
            conn.Open();
            using (MySqlCommand cmd = new MySqlCommand(query, conn))
            {
                cmd.Parameters.AddWithValue("@MaLop", maLop);
                cmd.Parameters.AddWithValue("@MaMonHoc", maMonHoc);
                cmd.Parameters.AddWithValue("@MaHocKy", maHocKy);

                if (maPhanCongExclude.HasValue)
                {
                    cmd.Parameters.AddWithValue("@MaPhanCongExclude", maPhanCongExclude.Value);
                }

                int count = Convert.ToInt32(cmd.ExecuteScalar());
                return count > 0; // Trả về true nếu đã tồn tại
            }
        }
    }
    catch (Exception ex)
    {
        Console.WriteLine($"Lỗi KiemTraTrungLapMonHoc: {ex.Message}");
        throw;
    }
}
```

**Giải thích**:
- **Ràng buộc nghiệp vụ**: Một lớp không thể có 2 phân công cùng môn trong cùng học kỳ
- **Nullable parameter**: `maPhanCongExclude` dùng khi đang cập nhật (bỏ qua bản ghi hiện tại)
- **Dynamic query**: Thêm điều kiện `AND MaPhanCong != ...` nếu đang cập nhật

**KiemTraGiaoVienChuyenMon() - Kiểm tra giáo viên có chuyên môn**

```csharp
public bool KiemTraGiaoVienChuyenMon(string maGiaoVien, int maMonHoc)
{
    // ✅ Updated: Query GiaoVien table directly using MaMonChuyenMon
    const string sql = @"
        SELECT COUNT(*) 
        FROM GiaoVien 
        WHERE MaGiaoVien = @MaGiaoVien 
        AND MaMonChuyenMon = @MaMonHoc";
    using (var conn = ConnectionDatabase.GetConnection())
    {
        conn.Open();
        using (var cmd = new MySqlCommand(sql, conn))
        {
            cmd.Parameters.AddWithValue("@MaGiaoVien", maGiaoVien);
            cmd.Parameters.AddWithValue("@MaMonHoc", maMonHoc);
            int count = Convert.ToInt32(cmd.ExecuteScalar());
            return count > 0;
        }
    }
}
```

**Giải thích**:
- **Ràng buộc nghiệp vụ**: Giáo viên chỉ được phân công dạy môn đúng chuyên môn
- **Direct query**: Kiểm tra trực tiếp trong bảng `GiaoVien` (không cần join)

---

### 2.3. Lớp BUS (Business Logic)

#### 2.3.1. Cấu trúc cơ bản
**Vị trí**: `BUS/BUSClass/PhanCongGiangDayBUS.cs`

**Mục đích**: Xử lý logic nghiệp vụ, validation, chuyển đổi dữ liệu.

#### 2.3.2. Validation và Business Rules

**ThemPhanCong() - Thêm phân công với validation**

```csharp
public bool ThemPhanCong(PhanCongGiangDayDTO phanCong)
{
    try
    {
        // ✅ Validation dữ liệu đầu vào
        if (phanCong.MaLop <= 0)
            throw new ArgumentException("Mã lớp không hợp lệ");

        if (string.IsNullOrWhiteSpace(phanCong.MaGiaoVien))
            throw new ArgumentException("Mã giáo viên không được để trống");

        if (phanCong.MaMonHoc <= 0)
            throw new ArgumentException("Mã môn học không hợp lệ");

        if (phanCong.MaHocKy <= 0)
            throw new ArgumentException("Mã học kỳ không hợp lệ");

        if (phanCong.NgayKetThuc <= phanCong.NgayBatDau)
            throw new ArgumentException("Ngày kết thúc phải sau ngày bắt đầu");

        // ✅ Kiểm tra giáo viên có chuyên môn phù hợp
        if (!phanCongDAO.KiemTraGiaoVienChuyenMon(phanCong.MaGiaoVien, phanCong.MaMonHoc))
        {
            throw new Exception("Giáo viên không có chuyên môn phù hợp để dạy môn học này!");
        }

        // ✅ FIX: Kiểm tra trùng lặp môn học cho lớp trong học kỳ (không phân biệt giáo viên)
        if (phanCongDAO.KiemTraTrungLapMonHoc(phanCong.MaLop, phanCong.MaMonHoc, phanCong.MaHocKy))
        {
            throw new Exception("Môn học này đã được phân công cho lớp trong học kỳ này!");
        }

        return phanCongDAO.ThemPhanCong(phanCong);
    }
    catch (Exception ex)
    {
        throw new Exception($"Lỗi khi thêm phân công: {ex.Message}", ex);
    }
}
```

**Giải thích**:
- **Multi-layer validation**: Kiểm tra ở nhiều tầng (BUS → DAO)
- **Business rules**: 
  - Giáo viên phải có chuyên môn đúng
  - Một lớp không thể có 2 phân công cùng môn trong cùng học kỳ
- **Exception wrapping**: Bọc exception gốc để thêm context

**ValidateAssignment() - Validation trước khi lưu (cho UI)**

```csharp
public AssignmentValidationResult ValidateAssignment(PhanCongGiangDayDTO phanCong, bool isUpdate = false)
{
    var result = new AssignmentValidationResult { IsValid = true };

    try
    {
        // Input validation
        if (phanCong.MaLop <= 0)
        {
            result.IsValid = false;
            result.Errors.Add("Mã lớp không hợp lệ");
        }

        // ... các validation khác ...

        // Business rule validation (only if basic validation passes)
        if (result.IsValid)
        {
            // Check teacher expertise
            if (!phanCongDAO.KiemTraGiaoVienChuyenMon(phanCong.MaGiaoVien, phanCong.MaMonHoc))
            {
                result.IsValid = false;
                result.Errors.Add("Giáo viên không có chuyên môn phù hợp để dạy môn học này!");
            }

            // ✅ CRITICAL: Check for duplicate subject-class-semester
            if (isUpdate)
            {
                if (phanCongDAO.KiemTraTrungLapMonHoc(phanCong.MaLop, phanCong.MaMonHoc, phanCong.MaHocKy, phanCong.MaPhanCong))
                {
                    result.IsValid = false;
                    result.Errors.Add("Môn học này đã được phân công cho lớp trong học kỳ này!");
                }
            }
            else
            {
                if (phanCongDAO.KiemTraTrungLapMonHoc(phanCong.MaLop, phanCong.MaMonHoc, phanCong.MaHocKy))
                {
                    result.IsValid = false;
                    result.Errors.Add("Môn học này đã được phân công cho lớp trong học kỳ này!");
                }
            }
        }
    }
    catch (Exception ex)
    {
        result.IsValid = false;
        result.Errors.Add($"Lỗi khi kiểm tra dữ liệu: {ex.Message}");
    }

    return result;
}
```

**Giải thích**:
- **Pre-validation pattern**: Kiểm tra trước khi người dùng submit form
- **Non-throwing validation**: Trả về kết quả thay vì throw exception (UX tốt hơn)
- **Errors và Warnings**: Phân biệt lỗi nghiêm trọng và cảnh báo

#### 2.3.3. Chuyển đổi dữ liệu (DTO ↔ ViewModel)

**ConvertToViewModel() - Chuyển DTO sang ViewModel**

```csharp
public List<PhanCongGiangDayViewModel> ConvertToViewModel(List<PhanCongGiangDayDTO> dsPhanCong)
{
    if (dsPhanCong == null || dsPhanCong.Count == 0)
        return new List<PhanCongGiangDayViewModel>();

    try
    {
        // ✅ Cache các lookup để tránh N+1 query
        var giaoVienBUS = new GiaoVienBUS();
        var monHocBUS = new MonHocBUS();
        var lopHocBUS = new LopHocBUS();
        var hocKyBUS = new HocKyBUS();

        var giaoVienCache = new Dictionary<string, string>();
        var monHocCache = new Dictionary<int, string>();
        var lopCache = new Dictionary<int, string>();
        var hocKyCache = new Dictionary<int, string>();

        // ✅ Load tất cả lookup một lần
        var uniqueGV = dsPhanCong.Select(pc => pc.MaGiaoVien).Distinct().ToList();
        var uniqueMH = dsPhanCong.Select(pc => pc.MaMonHoc).Distinct().ToList();
        var uniqueLop = dsPhanCong.Select(pc => pc.MaLop).Distinct().ToList();
        var uniqueHK = dsPhanCong.Select(pc => pc.MaHocKy).Distinct().ToList();

        // Cache giáo viên
        foreach (var maGV in uniqueGV)
        {
            if (!giaoVienCache.ContainsKey(maGV))
            {
                var gv = giaoVienBUS.LayGiaoVienTheoMa(maGV);
                giaoVienCache[maGV] = gv != null ? gv.HoTen : maGV;
            }
        }

        // Cache môn học, lớp, học kỳ tương tự...

        // ✅ Tạo danh sách ViewModel
        var viewModels = new List<PhanCongGiangDayViewModel>();
        foreach (PhanCongGiangDayDTO pc in dsPhanCong)
        {
            viewModels.Add(new PhanCongGiangDayViewModel
            {
                MaPhanCong = pc.MaPhanCong,
                GiaoVien = giaoVienCache.ContainsKey(pc.MaGiaoVien) ? giaoVienCache[pc.MaGiaoVien] : pc.MaGiaoVien,
                MonHoc = monHocCache.ContainsKey(pc.MaMonHoc) ? monHocCache[pc.MaMonHoc] : $"MH-{pc.MaMonHoc}",
                Lop = lopCache.ContainsKey(pc.MaLop) ? lopCache[pc.MaLop] : $"Lớp-{pc.MaLop}",
                HocKy = hocKyCache.ContainsKey(pc.MaHocKy) ? hocKyCache[pc.MaHocKy] : $"HK-{pc.MaHocKy}",
                ThoiGian = $"{pc.NgayBatDau:dd/MM/yyyy} - {pc.NgayKetThuc:dd/MM/yyyy}",
                // Lưu mã gốc
                MaGiaoVien = pc.MaGiaoVien,
                MaMonHoc = pc.MaMonHoc,
                MaLop = pc.MaLop,
                MaHocKy = pc.MaHocKy
            });
        }

        return viewModels;
    }
    catch (Exception ex)
    {
        throw new Exception($"Lỗi khi chuyển đổi sang ViewModel: {ex.Message}", ex);
    }
}
```

**Giải thích**:
- **N+1 Query Problem**: Nếu không cache, sẽ có N queries cho N phân công
- **Caching pattern**: Load tất cả lookup một lần, sau đó dùng cache
- **LINQ Distinct()**: Lấy danh sách unique IDs để query
- **Format string**: `$"{pc.NgayBatDau:dd/MM/yyyy}"` - Format ngày tháng

#### 2.3.4. Lọc và thống kê

**ApplyFilters() - Lọc phân công theo nhiều tiêu chí**

```csharp
public List<PhanCongGiangDayDTO> ApplyFilters(
    List<PhanCongGiangDayDTO> dsPhanCong, 
    PhanCongFilterCriteria filterCriteria,
    bool skipHocKyFilter = false)
{
    if (dsPhanCong == null || dsPhanCong.Count == 0)
        return dsPhanCong ?? new List<PhanCongGiangDayDTO>();

    if (filterCriteria == null)
        return dsPhanCong;

    try
    {
        var filtered = dsPhanCong.AsEnumerable();  // ✅ LINQ to Objects

        // Filter theo Học kỳ hoặc Năm học
        if (!skipHocKyFilter)
        {
            if (!string.IsNullOrEmpty(filterCriteria.MaNamHoc))
            {
                var hocKyDAO = new HocKyDAO();
                var dsHocKy = hocKyDAO.DocDSHocKy();
                var maHocKyTrongNam = dsHocKy
                    .Where(hk => hk.MaNamHoc == filterCriteria.MaNamHoc)  // ✅ LINQ Where
                    .Select(hk => hk.MaHocKy)  // ✅ LINQ Select
                    .ToList();
                
                if (maHocKyTrongNam.Count > 0)
                {
                    filtered = filtered.Where(pc => maHocKyTrongNam.Contains(pc.MaHocKy));  // ✅ LINQ Contains
                }
            }
        }

        // Filter theo Khối
        if (filterCriteria.Khoi.HasValue)
        {
            var lopDAO = new LopDAO();
            filtered = filtered.Where(pc =>
            {
                var lop = lopDAO.LayLopTheoId(pc.MaLop);
                if (lop != null)
                {
                    string tenLop = lop.tenLop?.Trim() ?? "";
                    if (tenLop.Length > 0 && char.IsDigit(tenLop[0]))
                    {
                        string khoiStr = new string(tenLop.TakeWhile(char.IsDigit).ToArray());  // ✅ LINQ TakeWhile
                        return int.TryParse(khoiStr, out int lopKhoi) && lopKhoi == filterCriteria.Khoi.Value;
                    }
                }
                return false;
            });
        }

        // Filter theo Lớp, Môn học tương tự...

        return filtered.ToList();  // ✅ Materialize query
    }
    catch (Exception ex)
    {
        Console.WriteLine($"Lỗi apply filters: {ex.Message}");
        return dsPhanCong;
    }
}
```

**Giải thích**:
- **LINQ to Objects**: Lọc dữ liệu trong memory (không query database)
- **Deferred execution**: `AsEnumerable()` và các LINQ operators không thực thi ngay
- **Materialization**: `ToList()` mới thực thi query và trả về kết quả
- **Lambda expressions**: `pc => maHocKyTrongNam.Contains(pc.MaHocKy)` - Anonymous function

**GetStatistics() - Thống kê phân công**

```csharp
public Dictionary<string, int> GetStatistics(int? maHocKyFilter = null)
{
    try
    {
        List<PhanCongGiangDayDTO> dsPhanCong = DocDSPhanCong();
        
        // Áp dụng filter học kỳ nếu có
        if (maHocKyFilter.HasValue)
        {
            dsPhanCong = dsPhanCong?.Where(pc => pc.MaHocKy == maHocKyFilter.Value).ToList();  // ✅ LINQ Where
        }
        
        int tongPhanCong = dsPhanCong?.Count ?? 0;
        int tongGiaoVien = dsPhanCong?.Select(pc => pc.MaGiaoVien).Distinct().Count() ?? 0;  // ✅ LINQ Distinct
        int tongMonHoc = dsPhanCong?.Select(pc => pc.MaMonHoc).Distinct().Count() ?? 0;
        int tongLopHoc = dsPhanCong?.Select(pc => pc.MaLop).Distinct().Count() ?? 0;

        return new Dictionary<string, int>
        {
            { "TongPhanCong", tongPhanCong },
            { "TongGiaoVien", tongGiaoVien },
            { "TongMonHoc", tongMonHoc },
            { "TongLopHoc", tongLopHoc }
        };
    }
    catch (Exception ex)
    {
        throw new Exception($"Lỗi khi tính thống kê: {ex.Message}", ex);
    }
}
```

**Giải thích**:
- **LINQ aggregation**: `Select().Distinct().Count()` - Lấy số lượng unique
- **Null-conditional operator**: `dsPhanCong?.Count ?? 0` - Xử lý null an toàn
- **Dictionary return**: Trả về structured data thay vì nhiều biến riêng lẻ

---

### 2.4. Services - Tự động phân công

#### 2.4.1. AssignmentAutoService
**Vị trí**: `BUS/Services/AssignmentAutoService.cs`

**Mục đích**: Tự động sinh đề xuất phân công dựa trên chuyên môn giáo viên và tải giảng dạy.

**GenerateAutoAssignments() - Sinh phân công tự động**

```csharp
public AutoAssignResult GenerateAutoAssignments(int hocKyId, AssignmentPolicy policy)
{
    var result = new AutoAssignResult();
    
    // ✅ KIỂM TRA HỌC KỲ CÓ THỂ CHỈNH SỬA KHÔNG
    if (SemesterHelper.IsPast(hocKyId))
    {
        result.IsReadOnly = true;
        result.SemesterStatus = SemesterHelper.GetStatus(hocKyId);
        result.Report.HardViolations++;
        result.Report.Messages.Add($"⚠ Học kỳ này đã kết thúc ({result.SemesterStatus}). Không thể tạo phân công mới!");
        return result;
    }
    
    var lopBus = new LopHocBUS();
    var monBus = new MonHocBUS();
    var pcBus = new PhanCongGiangDayBUS();

    var classes = lopBus.DocDSLop();
    var subjects = monBus.DocDSMH();	
    var current = pcBus.LayPhanCongTheoHocKy(hocKyId);

    var teacherToLoad = GetTeacherWeeklyLoad(hocKyId);  // Dictionary<MaGV, SoTiet>
    var subjectToTeachers = GetSubjectSpecialists();     // Dictionary<MaMon, List<MaGV>>

    foreach (var lop in classes)
    {
        string gvcn = GetGVCN(lop.maLop);  // Lấy giáo viên chủ nhiệm

        foreach (var mon in subjects)
        {
            int required = mon.soTiet;
            if (required <= 0) continue;

            // ✅ Kiểm tra đã có phân công chưa
            bool already = current.Any(x => x.MaLop == lop.maLop && x.MaMonHoc == mon.maMon && x.MaHocKy == hocKyId);  // ✅ LINQ Any
            if (already) continue;

            // ✅ Lấy danh sách giáo viên có chuyên môn
            var candidates = subjectToTeachers.ContainsKey(mon.maMon)
                ? subjectToTeachers[mon.maMon]
                : new List<string>();

            // ✅ B1: Ưu tiên GVCN (không kiểm tra giới hạn tải)
            if (!string.IsNullOrEmpty(gvcn) && candidates.Contains(gvcn))
            {
                result.Candidates.Add(new PhanCongCandidate
                {
                    MaLop = lop.maLop,
                    MaMonHoc = mon.maMon,
                    MaGiaoVien = gvcn,
                    SoTietTuan = required,
                    Score = policy.SpecialtyWeight + policy.PriorityWeight * 10,  // Điểm cao cho GVCN
                    Note = "GVCN"
                });
                // Cập nhật tải để cân bằng
                if (!teacherToLoad.ContainsKey(gvcn)) teacherToLoad[gvcn] = 0;
                teacherToLoad[gvcn] += required;
                continue;
            }

            // ✅ B2: Chọn GV khác (chỉ chọn GV có chuyên môn đúng)
            var scored = new List<(string gv, int score)>();
            foreach (var gv in candidates)
            {
                int load = teacherToLoad.ContainsKey(gv) ? teacherToLoad[gv] : 0;
                
                // ✅ Tính điểm: chuyên môn + cân bằng tải
                int score = policy.SpecialtyWeight + (policy.LoadBalanceWeight * Math.Max(0, 100 - load));

                // ✅ Bonus nếu GV đã dạy lớp này
                bool sameClassOfficial = current.Any(x => x.MaLop == lop.maLop && x.MaGiaoVien == gv);
                bool sameClassProposed = result.Candidates.Any(x => x.MaLop == lop.maLop && x.MaGiaoVien == gv);
                if (sameClassOfficial || sameClassProposed) score += policy.PriorityWeight * 3;
                
                scored.Add((gv, score));
            }

            if (scored.Count == 0)
            {
                result.Report.HardViolations++;
                result.Report.Messages.Add($"Không tìm được GV có chuyên môn phù hợp cho Lớp {lop.maLop}, Môn {mon.maMon} ({mon.tenMon}).");
                continue;
            }

            // ✅ Chọn GV có điểm cao nhất
            var best = scored.OrderByDescending(x => x.score).First();  // ✅ LINQ OrderByDescending
            result.Candidates.Add(new PhanCongCandidate
            {
                MaLop = lop.maLop,
                MaMonHoc = mon.maMon,
                MaGiaoVien = best.gv,
                SoTietTuan = required,
                Score = best.score
            });
            // Cập nhật tải
            if (!teacherToLoad.ContainsKey(best.gv)) teacherToLoad[best.gv] = 0;
            teacherToLoad[best.gv] += required;
        }
    }

    return result;
}
```

**Giải thích**:
- **Greedy algorithm**: Chọn giáo viên tốt nhất cho từng phân công
- **Scoring system**: Điểm = chuyên môn + cân bằng tải + ưu tiên lớp
- **LINQ Any()**: Kiểm tra có phần tử nào thỏa điều kiện không
- **LINQ OrderByDescending()**: Sắp xếp giảm dần theo điểm

**GetSubjectSpecialists() - Lấy danh sách giáo viên theo môn**

```csharp
private Dictionary<int, List<string>> GetSubjectSpecialists()
{
    var result = new Dictionary<int, List<string>>();
    var gvBus = new GiaoVienBUS();
    
    try
    {
        // ✅ Get all teachers with their specialties
        var teachers = gvBus.DocDSGiaoVien();
        
        foreach (var teacher in teachers)
        {
            // Check if teacher is active and has specialty
            if (teacher.TrangThai == "Đang giảng dạy" && teacher.MaMonChuyenMon.HasValue)
            {
                int mon = teacher.MaMonChuyenMon.Value;
                string gv = teacher.MaGiaoVien;
                
                if (!result.ContainsKey(mon)) 
                    result[mon] = new List<string>();
                    
                if (!result[mon].Contains(gv)) 
                    result[mon].Add(gv);
            }
        }
    }
    catch (Exception ex)
    {
        Console.WriteLine($"❌ Lỗi GetSubjectSpecialists: {ex.Message}");
    }
    
    return result;
}
```

**Giải thích**:
- **Dictionary grouping**: Nhóm giáo viên theo môn chuyên môn
- **Filtering**: Chỉ lấy giáo viên đang hoạt động và có chuyên môn

#### 2.4.2. AssignmentPersistService
**Vị trí**: `BUS/Services/AssignmentPersistService.cs`

**Mục đích**: Lưu trữ phân công tạm và chuyển sang chính thức.

**PersistTemporary() - Lưu phân công tạm**

```csharp
public void PersistTemporary(List<PhanCongCandidate> list, int hocKyId)
{
    if (list == null || list.Count == 0)
        throw new ArgumentException("Danh sách phân công trống!");
    
    if (hocKyId <= 0)
        throw new ArgumentException("Học kỳ không hợp lệ!");
    
    // ✅ Xóa dữ liệu tạm CỦA HỌC KỲ này (không xóa toàn bộ)
    const string clearSql = "DELETE FROM PhanCong_Temp WHERE MaHocKy = @MaHocKy";
    const string insertSql = @"INSERT INTO PhanCong_Temp(MaLop, MaGiaoVien, MaMonHoc, MaHocKy, SoTietTuan, Note)
        VALUES(@MaLop, @MaGiaoVien, @MaMonHoc, @MaHocKy, @SoTietTuan, @Note)";
    
    using (var conn = ConnectionDatabase.GetConnection())
    {
        conn.Open();
        using (var tx = conn.BeginTransaction())
        {
            try
            {
                // Xóa dữ liệu tạm của học kỳ này
                using (var clear = new MySqlCommand(clearSql, conn, tx))
                {
                    clear.Parameters.AddWithValue("@MaHocKy", hocKyId);
                    clear.ExecuteNonQuery();
                }
                
                // Insert dữ liệu mới
                foreach (var c in list)
                {
                    using (var cmd = new MySqlCommand(insertSql, conn, tx))
                    {
                        cmd.Parameters.AddWithValue("@MaLop", c.MaLop);
                        cmd.Parameters.AddWithValue("@MaGiaoVien", c.MaGiaoVien);
                        cmd.Parameters.AddWithValue("@MaMonHoc", c.MaMonHoc);
                        cmd.Parameters.AddWithValue("@MaHocKy", hocKyId);
                        cmd.Parameters.AddWithValue("@SoTietTuan", c.SoTietTuan);
                        cmd.Parameters.AddWithValue("@Note", string.IsNullOrEmpty(c.Note) ? (object)DBNull.Value : c.Note);
                        cmd.ExecuteNonQuery();
                    }
                }
                
                tx.Commit();
            }
            catch
            {
                tx.Rollback();
                throw;
            }
        }
    }
}
```

**Giải thích**:
- **Temp table pattern**: Lưu tạm vào `PhanCong_Temp` để người dùng xem xét trước khi chấp nhận
- **Transaction**: Đảm bảo xóa và insert cùng lúc
- **DBNull handling**: Xử lý null cho optional fields

**AcceptToOfficial() - Chuyển từ temp sang chính thức**

```csharp
public void AcceptToOfficial(int hocKyId)
{
    // ✅ KIỂM TRA HỌC KỲ
    if (SemesterHelper.IsPast(hocKyId))
    {
        throw new InvalidOperationException($"Không thể lưu phân công cho học kỳ đã kết thúc! Trạng thái: {SemesterHelper.GetStatus(hocKyId)}");
    }
    
    // ✅ FIX: Lấy thời gian từ HocKy, không phải CURDATE()
    const string insertSql = @"
        INSERT INTO PhanCongGiangDay(MaLop, MaGiaoVien, MaMonHoc, MaHocKy, NgayBatDau, NgayKetThuc)
        SELECT 
            t.MaLop, 
            t.MaGiaoVien, 
            t.MaMonHoc, 
            t.MaHocKy,
            hk.NgayBD,
            hk.NgayKT
        FROM PhanCong_Temp t
        INNER JOIN HocKy hk ON t.MaHocKy = hk.MaHocKy
        WHERE t.MaHocKy = @MaHocKy
        ON DUPLICATE KEY UPDATE
            NgayBatDau = VALUES(NgayBatDau),
            NgayKetThuc = VALUES(NgayKetThuc)";
    
    const string clearSql = "DELETE FROM PhanCong_Temp WHERE MaHocKy = @MaHocKy";
    
    using (var conn = ConnectionDatabase.GetConnection())
    {
        conn.Open();
        using (var tx = conn.BeginTransaction())
        {
            try
            {
                int rowsAffected = 0;
                using (var cmd = new MySqlCommand(insertSql, conn, tx))
                {
                    cmd.Parameters.AddWithValue("@MaHocKy", hocKyId);
                    rowsAffected = cmd.ExecuteNonQuery();
                }
                
                if (rowsAffected == 0)
                {
                    tx.Rollback();
                    throw new InvalidOperationException("❌ Không có dữ liệu nào được lưu!");
                }
                
                using (var clr = new MySqlCommand(clearSql, conn, tx))
                {
                    clr.Parameters.AddWithValue("@MaHocKy", hocKyId);
                    clr.ExecuteNonQuery();
                }
                
                tx.Commit();
            }
            catch (Exception ex)
            {
                tx.Rollback();
                throw;
            }
        }
    }
}
```

**Giải thích**:
- **INSERT ... SELECT**: Copy dữ liệu từ temp sang chính thức
- **INNER JOIN**: Lấy ngày bắt đầu/kết thúc từ bảng `HocKy`
- **ON DUPLICATE KEY UPDATE**: Xử lý trùng lặp (nếu có unique constraint)

---

### 2.5. GUI Layer

#### 2.5.1. Cấu trúc cơ bản
**Vị trí**: `GUI/GUIClass/PhanCongGiangDay/PhanCongGiangDay.cs`

**Mục đích**: Giao diện người dùng, xử lý sự kiện, binding dữ liệu.

#### 2.5.2. Load và hiển thị dữ liệu

**LoadData() - Tải và hiển thị phân công**

```csharp
private void LoadData()
{
    try
    {
        // ✅ Lấy filter criteria từ UI
        var filterCriteria = GetFilterCriteria();
        int? maHocKyFilter = GetSelectedHocKyId();

        // ✅ Sử dụng BUS để lấy ViewModel (đã convert và filter)
        var viewModels = phanCongBUS.GetFilteredViewModels(filterCriteria, maHocKyFilter);

        // ✅ Binding vào DataGridView
        bindingList = new BindingList<PhanCongGiangDayViewModel>(viewModels);
        tablePhanCongGiangDay.DataSource = bindingList;

        // ✅ Cập nhật thống kê
        LoadStatCards(maHocKyFilter);
    }
    catch (Exception ex)
    {
        MessageBox.Show($"Lỗi khi tải dữ liệu: {ex.Message}", "Lỗi",
            MessageBoxButtons.OK, MessageBoxIcon.Error);
    }
}
```

**Giải thích**:
- **BindingList**: Tự động cập nhật DataGridView khi dữ liệu thay đổi
- **Separation of concerns**: GUI chỉ gọi BUS, không trực tiếp gọi DAO

#### 2.5.3. Xử lý sự kiện

**BtnThemPhanCong_Click() - Thêm phân công mới**

```csharp
private void BtnThemPhanCong_Click(object sender, EventArgs e)
{
    try
    {
        // ✅ Mở form thêm phân công
        var formThem = new ThemPhanCongGiangDay();
        if (formThem.ShowDialog() == DialogResult.OK)
        {
            // ✅ Reload dữ liệu sau khi thêm
            LoadData();
            MessageBox.Show("Thêm phân công thành công!", "Thông báo",
                MessageBoxButtons.OK, MessageBoxIcon.Information);
        }
    }
    catch (Exception ex)
    {
        MessageBox.Show($"Lỗi khi thêm phân công: {ex.Message}", "Lỗi",
            MessageBoxButtons.OK, MessageBoxIcon.Error);
    }
}
```

**Giải thích**:
- **Modal dialog**: `ShowDialog()` - Chờ người dùng đóng form
- **DialogResult**: Kiểm tra kết quả để quyết định có reload không

---

## 3. MỨC ĐỘ LIÊN QUAN VÀ TƯƠNG TÁC

### 3.1. Liên quan với Giáo viên
- **Kiểm tra chuyên môn**: `KiemTraGiaoVienChuyenMon()` - Query bảng `GiaoVien`
- **Lấy danh sách giáo viên theo môn**: `GetGiaoVienByMonHoc()` - Filter theo `MaMonChuyenMon`
- **Khi xóa giáo viên**: Phải chuyển phân công sang giáo viên khác

### 3.2. Liên quan với Thời khóa biểu
- **Input cho TKB**: Phân công giảng dạy là dữ liệu đầu vào để sinh thời khóa biểu
- **GetBySemester()**: Thời khóa biểu gọi để lấy phân công theo học kỳ

### 3.3. Liên quan với Lớp học
- **Lọc theo lớp**: `LayPhanCongTheoLop()`
- **Kiểm tra trùng lặp**: Một lớp không thể có 2 phân công cùng môn trong cùng học kỳ

### 3.4. Liên quan với Môn học
- **Lấy số tiết**: `GetRequiredPeriods()` - Lấy `SoTiet` từ bảng `MonHoc`
- **Lọc theo môn**: `ApplyFilters()` với `MaMonHoc`

### 3.5. Liên quan với Học kỳ
- **Lọc theo học kỳ**: `LayPhanCongTheoHocKy()`
- **Kiểm tra học kỳ cho phép chỉnh sửa**: `KiemTraHocKyChoPhepChinhSua()` - Không cho sửa học kỳ đã kết thúc

---

## 4. CÁC KỸ THUẬT VÀ PATTERN ĐƯỢC SỬ DỤNG

### 4.1. LINQ (Language Integrated Query)

#### 4.1.1. LINQ to Objects
**Ví dụ 1: Filter và Select**

```csharp
var filtered = dsPhanCong
    .Where(pc => pc.MaHocKy == maHocKyFilter.Value)  // Filter
    .Select(pc => pc.MaGiaoVien)                    // Projection
    .Distinct()                                      // Remove duplicates
    .ToList();                                       // Materialize
```

**Giải thích**:
- **Where()**: Lọc các phần tử thỏa điều kiện (tương đương SQL WHERE)
- **Select()**: Chuyển đổi mỗi phần tử (tương đương SQL SELECT)
- **Distinct()**: Loại bỏ trùng lặp (tương đương SQL DISTINCT)
- **ToList()**: Thực thi query và trả về List

**Ví dụ 2: GroupBy và Aggregation**

```csharp
var statistics = dsPhanCong
    .GroupBy(pc => pc.MaGiaoVien)           // Nhóm theo giáo viên
    .Select(g => new {
        MaGV = g.Key,
        SoPhanCong = g.Count()              // Đếm số phân công
    })
    .OrderByDescending(x => x.SoPhanCong)   // Sắp xếp giảm dần
    .ToList();
```

**Giải thích**:
- **GroupBy()**: Nhóm các phần tử theo key (tương đương SQL GROUP BY)
- **Count()**: Đếm số phần tử trong mỗi nhóm
- **OrderByDescending()**: Sắp xếp giảm dần

**Ví dụ 3: Any() và Contains()**

```csharp
// Kiểm tra đã có phân công chưa
bool already = current.Any(x => 
    x.MaLop == lop.maLop && 
    x.MaMonHoc == mon.maMon && 
    x.MaHocKy == hocKyId);

// Kiểm tra giáo viên có trong danh sách không
if (candidates.Contains(gvcn))
{
    // ...
}
```

**Giải thích**:
- **Any()**: Trả về true nếu có ít nhất 1 phần tử thỏa điều kiện
- **Contains()**: Kiểm tra phần tử có trong collection không

#### 4.1.2. LINQ với Dictionary

```csharp
// Cache để tránh N+1 query
var giaoVienCache = new Dictionary<string, string>();

// Load tất cả unique giáo viên
var uniqueGV = dsPhanCong
    .Select(pc => pc.MaGiaoVien)
    .Distinct()
    .ToList();

// Populate cache
foreach (var maGV in uniqueGV)
{
    if (!giaoVienCache.ContainsKey(maGV))
    {
        var gv = giaoVienBUS.LayGiaoVienTheoMa(maGV);
        giaoVienCache[maGV] = gv != null ? gv.HoTen : maGV;
    }
}

// Sử dụng cache
string tenGV = giaoVienCache.ContainsKey(pc.MaGiaoVien) 
    ? giaoVienCache[pc.MaGiaoVien] 
    : pc.MaGiaoVien;
```

**Giải thích**:
- **Dictionary lookup**: O(1) thay vì O(N) khi tìm kiếm
- **Cache pattern**: Tránh query database nhiều lần

### 4.2. Transaction Pattern

```csharp
using (var conn = ConnectionDatabase.GetConnection())
{
    conn.Open();
    using (var tx = conn.BeginTransaction())
    {
        try
        {
            // Multiple operations
            cmd1.ExecuteNonQuery();
            cmd2.ExecuteNonQuery();
            
            tx.Commit();  // ✅ Commit nếu thành công
        }
        catch
        {
            tx.Rollback();  // ✅ Rollback nếu có lỗi
            throw;
        }
    }
}
```

**Giải thích**:
- **ACID properties**: Đảm bảo tính nhất quán dữ liệu
- **All or nothing**: Hoặc tất cả thành công, hoặc tất cả rollback

### 4.3. Repository Pattern (DAO)

```csharp
public class PhanCongGiangDayDAO
{
    // CRUD operations
    public List<PhanCongGiangDayDTO> DocDSPhanCong() { ... }
    public bool ThemPhanCong(PhanCongGiangDayDTO phanCong) { ... }
    public bool CapNhatPhanCong(PhanCongGiangDayDTO phanCong) { ... }
    public bool XoaPhanCong(int maPhanCong) { ... }
}
```

**Giải thích**:
- **Separation of concerns**: Tách biệt data access và business logic
- **Testability**: Dễ mock DAO để test BUS

### 4.4. DTO Pattern

```csharp
// DTO cho nghiệp vụ
public class PhanCongGiangDayDTO { ... }

// ViewModel cho hiển thị
public class PhanCongGiangDayViewModel { ... }
```

**Giải thích**:
- **Data transfer**: Chuyển dữ liệu giữa các tầng
- **View model**: Tách biệt dữ liệu hiển thị và dữ liệu nghiệp vụ

---

## 5. LUỒNG XỬ LÝ ĐIỂN HÌNH

### 5.1. Thêm phân công mới

```
1. User click "Thêm phân công" trong GUI
   ↓
2. GUI mở form ThemPhanCongGiangDay
   ↓
3. User nhập thông tin và click "Lưu"
   ↓
4. GUI gọi phanCongBUS.ThemPhanCong(dto)
   ↓
5. BUS validate dữ liệu:
   - Kiểm tra mã hợp lệ
   - Kiểm tra giáo viên có chuyên môn
   - Kiểm tra trùng lặp môn học
   ↓
6. BUS gọi phanCongDAO.ThemPhanCong(dto)
   ↓
7. DAO mở transaction:
   - Kiểm tra trùng lặp (KiemTraTrungLapMonHoc)
   - INSERT vào PhanCongGiangDay
   - Commit transaction
   ↓
8. Trả về kết quả cho GUI
   ↓
9. GUI reload dữ liệu và hiển thị thông báo thành công
```

### 5.2. Tự động phân công

```
1. User click "Tự động phân công" trong GUI
   ↓
2. GUI gọi AssignmentAutoService.GenerateAutoAssignments()
   ↓
3. Service load dữ liệu:
   - Danh sách lớp học
   - Danh sách môn học
   - Phân công hiện tại
   - Danh sách giáo viên theo chuyên môn
   ↓
4. Với mỗi (Lớp, Môn):
   - Kiểm tra đã có phân công chưa
   - Ưu tiên GVCN nếu có
   - Tính điểm cho các GV có chuyên môn:
     * Điểm = Chuyên môn + Cân bằng tải + Ưu tiên lớp
   - Chọn GV có điểm cao nhất
   ↓
5. Trả về danh sách PhanCongCandidate
   ↓
6. GUI hiển thị preview, user có thể chỉnh sửa
   ↓
7. User click "Lưu tạm" → PersistTemporary()
   ↓
8. User click "Chấp nhận" → AcceptToOfficial()
```

### 5.3. Lọc và tìm kiếm

```
1. User chọn filter (Học kỳ, Khối, Lớp, Môn)
   ↓
2. GUI gọi GetFilterCriteria() để lấy criteria
   ↓
3. GUI gọi phanCongBUS.GetFilteredViewModels(criteria)
   ↓
4. BUS:
   - Lấy dữ liệu từ DAO (có thể filter ở DB level)
   - Áp dụng LINQ filters trong memory
   - Convert sang ViewModel với caching
   ↓
5. Trả về ViewModel list
   ↓
6. GUI bind vào DataGridView
```

---

## 6. VÍ DỤ THỰC TẾ

### 6.1. Ví dụ: Thêm phân công cho lớp 10A1, môn Toán, học kỳ 1

**Input**:
- MaLop = 1 (10A1)
- MaMonHoc = 1 (Toán)
- MaGiaoVien = "GV001" (Nguyễn Văn A)
- MaHocKy = 1

**Quy trình**:

1. **GUI**: User nhập form và click "Lưu"
2. **BUS Validation**:
   ```csharp
   // Kiểm tra giáo viên có chuyên môn Toán không
   if (!phanCongDAO.KiemTraGiaoVienChuyenMon("GV001", 1))
       throw new Exception("Giáo viên không có chuyên môn Toán!");
   
   // Kiểm tra lớp 10A1 đã có phân công Toán trong HK1 chưa
   if (phanCongDAO.KiemTraTrungLapMonHoc(1, 1, 1))
       throw new Exception("Lớp 10A1 đã có phân công Toán trong HK1!");
   ```

3. **DAO Insert**:
   ```sql
   INSERT INTO PhanCongGiangDay(MaLop, MaGiaoVien, MaMonHoc, MaHocKy, NgayBatDau, NgayKetThuc)
   VALUES(1, 'GV001', 1, 1, '2024-09-01', '2024-12-31')
   ```

### 6.2. Ví dụ: Tự động phân công cho tất cả lớp

**Input**: Học kỳ 1, Policy (MaxLoad = 100, AllowNonPrimary = false)

**Quy trình**:

1. **Load dữ liệu**:
   - 20 lớp (10A1, 10A2, ..., 12C5)
   - 15 môn học (Toán, Văn, Anh, ...)
   - 30 giáo viên

2. **Với mỗi (Lớp, Môn)**:
   ```
   Lớp 10A1, Môn Toán:
   - GVCN = "GV001" (có chuyên môn Toán) → Chọn GV001, Score = 15
   
   Lớp 10A1, Môn Văn:
   - GVCN = "GV001" (không có chuyên môn Văn)
   - Candidates = ["GV005", "GV012"] (có chuyên môn Văn)
   - GV005: Load = 0 → Score = 5 + (3 * 100) = 305
   - GV012: Load = 50 → Score = 5 + (3 * 50) = 155
   - Chọn GV005 (điểm cao hơn)
   ```

3. **Kết quả**: 300 phân công được đề xuất

---

## 7. TỐI ƯU HÓA VÀ BEST PRACTICES

### 7.1. Tránh N+1 Query Problem

**❌ Không tốt**:
```csharp
foreach (var pc in dsPhanCong)
{
    var gv = giaoVienBUS.LayGiaoVienTheoMa(pc.MaGiaoVien);  // Query trong loop!
    // ...
}
```

**✅ Tốt**:
```csharp
// Load tất cả một lần
var uniqueGV = dsPhanCong.Select(pc => pc.MaGiaoVien).Distinct().ToList();
var giaoVienCache = new Dictionary<string, string>();
foreach (var maGV in uniqueGV)
{
    var gv = giaoVienBUS.LayGiaoVienTheoMa(maGV);
    giaoVienCache[maGV] = gv.HoTen;
}

// Sử dụng cache
foreach (var pc in dsPhanCong)
{
    string tenGV = giaoVienCache[pc.MaGiaoVien];  // O(1) lookup
}
```

### 7.2. Sử dụng Transaction cho Batch Operations

**✅ Tốt**:
```csharp
using (var tx = conn.BeginTransaction())
{
    foreach (var item in list)
    {
        InsertItem(item, conn, tx);  // Shared transaction
    }
    tx.Commit();  // Commit một lần
}
```

### 7.3. Parameterized Queries

**❌ Không an toàn**:
```csharp
string query = $"SELECT * FROM PhanCongGiangDay WHERE MaHocKy = {maHocKy}";
```

**✅ An toàn**:
```csharp
string query = "SELECT * FROM PhanCongGiangDay WHERE MaHocKy = @MaHocKy";
cmd.Parameters.AddWithValue("@MaHocKy", maHocKy);
```

---

## 8. KẾT LUẬN

Module **Phân Công Giảng Dạy** là module trung tâm kết nối các module khác trong hệ thống. Nó sử dụng:

- **3-layer architecture**: GUI → BUS → DAO
- **DTO/ViewModel pattern**: Tách biệt dữ liệu nghiệp vụ và hiển thị
- **LINQ**: Xử lý dữ liệu trong memory hiệu quả
- **Transaction**: Đảm bảo tính nhất quán dữ liệu
- **Caching**: Tránh N+1 query problem
- **Validation**: Multi-layer validation (BUS + DAO)

Module này cung cấp nền tảng cho việc sinh thời khóa biểu tự động và quản lý phân công giảng dạy một cách hiệu quả.
